# Benchmark phát hiện URL lừa đảo: làm lại từng bước và kiểm mọi con số

Notebook này làm lại toàn bộ thí nghiệm của bài về việc một benchmark phát hiện URL lừa đảo thực sự đo cái gì: bảy họ mô hình trên bộ PhishVN (53.116 URL),
với chia ngẫu nhiên và chia theo thời gian, cấu hình huấn luyện, dòng trùng lặp, chuyển giao giữa bốn corpus, và một mạng CNN trên ký tự. Mọi phép tính nằm
ngay trong notebook. Notebook chỉ chạy, in kết quả, và so kết quả với bản kỳ vọng mà bài đã dùng.

### Cách chạy

1. Tải thư mục chứa notebook này (gồm `url_benchmark_walkthrough.ipynb`, `expected/` và `requirements.txt`).
2. Cài thư viện: `pip install -r requirements.txt` (Python 3.12). Các phiên bản được ghim vì kết quả phụ thuộc vào chúng, kể cả danh sách hậu tố tên miền
   đóng gói trong `tldextract`.
3. Tải gói dữ liệu ba corpus ngoài từ Zenodo (DOI ở ô đầu tiên) và giải nén vào thư mục `url_data/` cạnh notebook, hoặc để ô đầu tự tải. Bộ PhishVN
   được tải tự động từ Mendeley Data.
4. Mở notebook và chạy hết (Run All). Toàn bộ mất khoảng mười tiếng trên CPU, phần lớn là các bước huấn luyện CNN (Bước 20 tới 22).

### Kiểm như thế nào

Mỗi thí nghiệm ghi một bảng kết quả, mỗi dòng một lần chạy (họ mô hình, giao thức, seed…). Notebook so từng dòng với bảng kỳ vọng theo các cột khoá, bỏ qua
cột thời gian chạy (đồng hồ thật, không lặp lại được). Các số đo được so theo dung sai `MET_TOL` = 10⁻⁴, chặt hơn ba chữ số thập phân mà bài in: các họ
cây (random forest, XGBoost, LightGBM) huấn luyện song song trên nhiều luồng, nên thứ tự cộng số thực, và do đó những chữ số cuối, khác nhau giữa hai lần chạy
(thường lệch cỡ 10⁻⁷ tới 10⁻⁶; một dòng test nằm sát ngưỡng 0,5 mà đổi phía thì F1 lệch cỡ 10⁻⁵). Ma trận chuyển giao làm tròn ba chữ số nên được so đúng
từng ô. Với mỗi phép so, notebook in mức lệch lớn nhất thật sự.

Có hai ngoại lệ, đều nói rõ ở bước tương ứng: mạng CNN của ma trận chuyển giao và của phép thử khử trùng lặp được bài huấn luyện trên GPU của Apple (MPS),
còn notebook huấn luyện trên CPU, nên chỉ so được theo dung sai rộng hơn; và các con số chỉ là thời gian chạy (chi phí của các ensemble) không được so.

| Bước | Làm gì |
|---|---|
| 0 | thiết lập, tải dữ liệu, các hàm kiểm |
| 1 | mã dùng chung: đặc trưng, mô hình, chỉ số, cách chia theo thời gian |
| 2 | benchmark bảy họ, chia ngẫu nhiên và chia chính thức |
| 3 | chia theo thời gian nghiêm ngặt và đối chứng ngẫu nhiên trên cùng các dòng |
| 4 | đối chứng có guard tên miền, khớp tỷ lệ lớp, chia benign theo tên miền |
| 5 | CatBoost 20 seed |
| 6 | cấu hình cũ có trọng số lớp (để đối chiếu) |
| 7 | tách ảnh hưởng của tỷ lệ lớp và của tập dòng |
| 8 | ngưỡng chọn trên tập validation |
| 9 | chia theo thời gian cho cả hai lớp |
| 10 | dòng trùng và rò rỉ giữa train và test |
| 11 | ma trận chuyển giao giữa bốn corpus và các cách sửa |
| 12 | huấn luyện gộp nhiều corpus |
| 13 | khoảng tin cậy bootstrap của ma trận chuyển giao |
| 14 | SHAP và trọng số lớp |
| 15 | tối ưu siêu tham số (GWO và ngẫu nhiên) |
| 16 | stacking |
| 17 | khả năng dự báo độ trôi theo thời gian |
| 18 | kiểm toán nhãn nhiễu |
| 19 | độ nhạy theo tầng nhãn |
| 20 | CNN trên ký tự, và phép thử nguồn benign |
| 21 | CNN trên ký tự trong ma trận chuyển giao |
| 22 | bỏ dòng trùng và chạy lại |
| 23 | bỏ tiền tố `www.` |
| 24 | các con số tính từ kết quả: kiểm định, bootstrap, đếm lại |
| 25 | tổng hợp |

## Bước 0: Thiết lập và tải dữ liệu

Mã của bài đọc và ghi theo đường dẫn tương đối `data/processed/...`. Ô đầu dựng đúng cây thư mục đó trong `url_work/` và chuyển thư mục làm việc vào đó:
- `data/processed/dataset_url.csv` và `vn_compphish.csv` (bộ PhishVN và 21 đặc trưng URL của nó), lấy từ gói `PhishVN_v3.1.0_open.zip` trên Mendeley Data;
- `data/processed/external/` ba corpus ngoài từ gói Zenodo;
- `data/processed/first_seen_validation_summary.json`, bản tóm tắt kiểm chứng ngày phát hiện, đi kèm trong `expected/`.

Mọi kết quả notebook tạo ra nằm trong `url_work/data/processed/p2/`.

In [ ]:
import os, re, io, csv, sys, json, math, time, random, shutil, hashlib, zipfile, itertools, collections, warnings, urllib.request
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

MENDELEY_ZIP = ("https://data.mendeley.com/public-files/datasets/b97hxbxtpd/files/"
                "d5724d4d-9c2b-46b7-810e-cf290c3bfd87/file_downloaded")
ZIP_SHA256 = "308351e9d0c0fca13a81f2f63524a7dee08ccba6c775848d125b3da5525d0ab5"
PHISHVN_SHA256 = {"dataset_url.csv": "8e1c6325a82177014ac892a9aaf8c4d3cbbc986b47fd1ecac8899c865fb6eade",
                  "vn_compphish.csv": "ce64567f4ab955b2"}
ZENODO_DOI = "10.5281/zenodo.XXXXXXX"
ZENODO_RECORD = None  # the record number, e.g. 1234567; filled in when the deposit is published
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/url_benchmark/expected"

HOME = Path.cwd()
WORK = HOME / "url_work"
PROC = WORK / "data" / "processed"
(PROC / "external").mkdir(parents=True, exist_ok=True)
(PROC / "p2").mkdir(parents=True, exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

def fetch(url, dest):
    # Mendeley refuses Python's default user agent
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (PhishVN walkthrough notebook)"})
    Path(dest).write_bytes(urllib.request.urlopen(req, timeout=300).read())

zip_path = WORK / "PhishVN_v3.1.0_open.zip"
if not zip_path.exists():
    fetch(MENDELEY_ZIP, zip_path)
assert sha(zip_path) == ZIP_SHA256, "the downloaded package is not the published one"
with zipfile.ZipFile(zip_path) as z:
    for name in PHISHVN_SHA256:
        (PROC / name).write_bytes(z.read(f"data/{name}"))
for name, digest in PHISHVN_SHA256.items():
    assert sha(PROC / name).startswith(digest), name

DATA = HOME / "url_data"
if not (DATA / "SHA256SUMS").exists():
    assert ZENODO_RECORD, f"download the data package (DOI {ZENODO_DOI}) and unzip it into {DATA}"
    DATA.mkdir(exist_ok=True)
    meta = json.loads(urllib.request.urlopen(f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=120).read())
    for f in meta["files"]:
        fetch(f["links"]["self"], DATA / f["key"])
for line in (DATA / "SHA256SUMS").read_text().splitlines():
    digest, name = line.split()
    assert sha(DATA / name) == digest, f"{name} differs from the published package"
    if name.endswith("_compphish.csv"):
        shutil.copyfile(DATA / name, PROC / "external" / name)

EXPECTED = HOME / "expected"
if not (EXPECTED / "manifest.json").exists():
    EXPECTED = WORK / "expected"
    EXPECTED.mkdir(exist_ok=True)
    fetch(f"{EXPECTED_URL}/manifest.json", EXPECTED / "manifest.json")
    for f in json.loads((EXPECTED / "manifest.json").read_text()):
        (EXPECTED / f).parent.mkdir(parents=True, exist_ok=True)
        if not (EXPECTED / f).exists():
            fetch(f"{EXPECTED_URL}/{f}", EXPECTED / f)
shutil.copyfile(EXPECTED / "first_seen_validation_summary.json", PROC / "first_seen_validation_summary.json")

os.chdir(WORK)
ROOT = str(WORK)                       # the code below joins its paths onto ROOT
os.environ["P2_UNWEIGHTED"] = "1"      # a few functions read the weighting switch from the environment
RES = "data/processed/p2"
print("working in", WORK, "| expected:", EXPECTED)

Các hàm kiểm:

- `check_table(step, name, tol)` so file `data/processed/p2/<name>` notebook vừa ghi với bản kỳ vọng cùng tên. Dòng được ghép theo mọi cột không phải số
  thực (họ, giao thức, seed, arm…); cột thời gian chạy bị bỏ qua; mọi cột số thực phải lệch không quá `tol`. Số dòng và các khoá phải trùng tuyệt đối.
- `check_matrix(step, name, tol)` so một ma trận (hàng là corpus huấn luyện, cột là corpus kiểm tra) từng ô.
- `check_json(step, mine, saved_name, key=None, tol=0)` so mọi trường của một kết quả với một file JSON kỳ vọng.
- `report(step)` in số phép khớp của một bước.

In [ ]:
CHECKS = []
TIME_COLS = {"fit_seconds", "seconds", "fit_s", "time_s", "elapsed_s", "wall_seconds", "fit_time"}
MET_TOL = 1e-4     # metrics of one run: parallel tree ensembles move the last digits, and one test row at
                   # the 0.5 threshold can flip; the paper prints three decimals
CURVE_TOL = 1e-4   # seed-mean precision on the 200-point recall grid, rounded to 6 decimals

def check(step, name, ok, detail=""):
    CHECKS.append({"step": step, "check": name, "detail": detail, "match": bool(ok)})
    return ok

def report(step):
    rows = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows if not c["match"]]
    print(f"[{step}] {len(rows) - len(bad)}/{len(rows)} checks match")
    for c in rows:
        print(f"    {'ok      ' if c['match'] else 'MISMATCH'} {c['check']}: {c['detail']}")

def _num_diff(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    both = np.isnan(x) & np.isnan(y)
    one = np.isnan(x) ^ np.isnan(y)
    d = np.where(both, 0.0, np.abs(x - y))
    return (np.inf if one.any() else float(np.nanmax(d))) if d.size else 0.0

def check_table(step, name, tol, path=None, col_tol=None, shared_columns=False):
    # col_tol: a wider tolerance for named columns; shared_columns: compare on the columns both
    # tables have (for a saved table written before the code added a column)
    new = pd.read_csv(path or f"{RES}/{name}")
    old = pd.read_csv(EXPECTED / name)
    extra = sorted(set(new.columns) - set(old.columns))
    if shared_columns and extra and not set(old.columns) - set(new.columns):
        new = new[list(old.columns)]
    if set(new.columns) != set(old.columns) or len(new) != len(old):
        return check(step, name, False, f"shape/columns differ: {new.shape} {sorted(set(new.columns) ^ set(old.columns))} vs {old.shape}")
    cols = [c for c in old.columns if c not in TIME_COLS]
    num = [c for c in cols if pd.api.types.is_float_dtype(old[c])]
    key = [c for c in cols if c not in num]
    # stable sort: rows sharing a key (a PR curve's 200 points) keep their order
    a = new[cols].sort_values(key, kind="mergesort").reset_index(drop=True) if key else new[cols]
    b = old[cols].sort_values(key, kind="mergesort").reset_index(drop=True) if key else old[cols]
    if key and not a[key].astype(str).equals(b[key].astype(str)):
        return check(step, name, False, "row keys differ")
    col_tol = col_tol or {}
    plain = [c for c in num if c not in col_tol]
    d = _num_diff(a[plain].values, b[plain].values) if plain else 0.0
    ok = d <= tol
    detail = f"{len(new)} rows, max |difference| {d:.2g} (tolerance {tol:g})"
    for c, t in col_tol.items():
        dc = _num_diff(a[[c]].values, b[[c]].values)
        ok &= dc <= t
        detail += f"; {c} {dc:.2g} (tolerance {t:g})"
    if shared_columns and extra:
        detail += f"; columns only in the notebook's table: {extra}"
    return check(step, name, ok, detail)

def check_matrix(step, name, tol=0.0, path=None):
    new = pd.read_csv(path or f"{RES}/{name}", index_col=0)
    old = pd.read_csv(EXPECTED / name, index_col=0)
    if new.shape != old.shape or list(new.index) != list(old.index) or list(new.columns) != list(old.columns):
        return check(step, name, False, "shape or corpus order differs")
    d = _num_diff(new.values, old.values)
    return check(step, name, d <= tol + 1e-12, f"{new.shape[0]}x{new.shape[1]}, max |difference| {d:.2g} (tolerance {tol:g})")

def _plain(o):
    def conv(v):
        if hasattr(v, "item"):
            return v.item()
        if hasattr(v, "tolist"):
            return v.tolist()
        return str(v)
    return json.loads(json.dumps(o, sort_keys=True, default=conv, allow_nan=True))

def flat(d, prefix=""):
    if isinstance(d, dict):
        for k, v in d.items():
            yield from flat(v, f"{prefix}.{k}" if prefix else str(k))
    elif isinstance(d, list):
        for i, v in enumerate(d):
            yield from flat(v, f"{prefix}[{i}]")
    else:
        yield prefix, d

def check_json(step, mine, saved_name, key=None, tol=0.0):
    saved = json.loads((EXPECTED / saved_name).read_text())
    new, ref = dict(flat(_plain(mine))), dict(flat(saved if key is None else saved[key]))
    keys = sorted(set(new) | set(ref))
    bad, worst = [], 0.0
    for k in keys:
        a, b = new.get(k, "<missing>"), ref.get(k, "<missing>")
        if isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            d = 0.0 if (a == b or (math.isnan(a) and math.isnan(b))) else abs(a - b)
            worst = max(worst, d)
            if not d <= tol:
                bad.append(k)
        elif a != b:
            bad.append(k)
    label = f"{saved_name}[{key}]" if key else saved_name
    check(step, label, not bad, f"{len(keys)} fields, {len(bad)} differ, max |difference| {worst:.2g} (tolerance {tol:g})")
    for k in bad[:8]:
        print(f"    differs: {k}: saved={ref.get(k, '<missing>')!r} notebook={new.get(k, '<missing>')!r}")

## Bước 1: Mã dùng chung

**Đặc trưng.** Mỗi URL được mô tả bằng 21 đặc trưng từ vựng theo lược đồ CompPhish (`COMPPHISH`): độ dài, số chữ, số chữ số, số ký tự đặc biệt, số dấu chấm,
số tên miền con, độ dài đuôi tên miền, entropy… Trong `vn_compphish.csv` và ba corpus ngoài, đặc trưng đã được tính sẵn.

**Mô hình.** Bảy họ: hồi quy logistic, random forest, HistGB, MLP (`make_model`), cộng XGBoost, LightGBM và CatBoost (`make_any_model`), đều với tham số mặc
định ghi trong mã. Từ bản sửa của bài, mọi họ được huấn luyện **không** cân trọng số lớp (`weighting_params` tắt `class_weight="balanced"` mà ba họ đầu kế
thừa); cấu hình cũ, trộn có và không trọng số, chỉ chạy lại ở Bước 6 để đối chiếu.

**Chỉ số.** F1 tại ngưỡng 0,5; PR-AUC; ROC-AUC; và FPR@R0.90, tỷ lệ báo nhầm khi bắt được 90% URL lừa đảo (`_metrics`). Mỗi lần chạy còn lưu một đường
precision-recall trên lưới recall cố định 200 điểm (`pr_curve_row`); `write_curves` lấy trung bình theo seed.

**Chia theo thời gian.** `load` ghép đặc trưng với ngày phát hiện trong `dataset_url.csv` và tính tên miền đăng ký (`registered_domain`, theo danh sách hậu tố
đóng gói trong `tldextract`, không tải mạng). `split_phishing` lấy 70% URL lừa đảo có ngày sớm nhất làm train, phần còn lại làm test, và bỏ khỏi test mọi URL có
tên miền đăng ký đã xuất hiện trong train (guard). `benign_mask` chia lớp benign ngẫu nhiên theo seed (hoặc theo tên miền nếu `grouped`); `match_prior` lấy mẫu
bớt benign để tỷ lệ lớp của tập test bằng một giá trị cho trước.

**Kiểm định.** So hai cấu hình trên cùng các seed dùng t hiệu chỉnh của Nadeau và Bengio: các lần chia train/test chồng lên nhau nên sai số chuẩn được nới thành
`sqrt((1/K + n_test/n_train) Var(d))`, với `d` là hiệu theo từng seed (`corrected_paired_t`). Khi có nhiều phép so, p được hiệu chỉnh Benjamini-Hochberg
(`bh_adjust`).

In [ ]:
import tldextract
_EXTRACT = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=True)


_TWO_LEVEL = {"com.vn", "net.vn", "org.vn", "edu.vn", "gov.vn", "ac.vn", "info.vn", "pro.vn",
              "health.vn", "int.vn", "name.vn", "biz.vn", "co.uk", "com.br", "com.au", "co.jp",
              "pages.dev", "workers.dev", "r2.dev", "netlify.app", "vercel.app", "web.app",
              "firebaseapp.com", "amplifyapp.com", "webflow.io", "weebly.com", "wixsite.com",
              "blogspot.com", "github.io", "duckdns.org", "ddns.net", "glitch.me", "repl.co"}


def apex(result) -> str:
    new = getattr(result, "top_domain_under_public_suffix", None)
    return result.registered_domain if new is None else new


def registered_domain(host: str) -> str:
    if _EXTRACT is not None:
        try:
            return apex(_EXTRACT(host)).lower() or host
        except Exception:
            pass
    parts = host.split(".")
    if len(parts) >= 3 and ".".join(parts[-2:]) in _TWO_LEVEL:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:]) if len(parts) >= 2 else host


# every library the steps below use
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)
from sklearn.model_selection import (train_test_split, GroupShuffleSplit, StratifiedKFold, TimeSeriesSplit,
                                     cross_val_predict, cross_val_score)
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits


MODEL_NAMES = ["LogReg", "RandomForest", "HistGB", "MLP"]


DETERMINISTIC = {"LogReg"}  # no seed effect; reported once


COMPPHISH = ["url_len", "dom_len", "is_ip", "tld_len", "subdom_cnt", "letter_cnt", "digit_cnt",
             "special_cnt", "eq_cnt", "qm_cnt", "amp_cnt", "dot_cnt", "dash_cnt", "under_cnt",
             "letter_ratio", "digit_ratio", "spec_ratio", "slash_cnt", "entropy",
             "path_len", "query_len"]


def add_label(df):
    # Unknown is not a negative. Respect explicit training eligibility and reject
    # unresolved labels instead of silently converting them to benign.
    if "training_eligible" in df:
        eligible = df["training_eligible"].astype(str).str.lower().isin({"1", "true"})
        if not eligible.all():
            raise ValueError("observations excluded by the selected label policy cannot be used for training/evaluation")
    lab = df["label"]
    if lab.dtype != object:
        numeric = pd.to_numeric(lab, errors="coerce")
        if not numeric.isin([0, 1]).all():
            raise ValueError("labels must be known binary outcomes, not missing or unknown")
        df["y"] = numeric.astype(int)
    else:
        pos = {"phishing", "phish", "spam", "smishing", "malicious", "1"}
        negative = {"benign", "legitimate", "ham", "0"}
        normalized = lab.astype(str).str.strip().str.lower()
        if not normalized.isin(pos | negative).all():
            raise ValueError("unknown label cannot be silently converted to benign")
        df["y"] = normalized.isin(pos).astype(int)
    return df


def fpr_at_recall(y, score, target_recall=0.90):
    if len(set(y)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(y, score)
    idx = np.where(tpr >= target_recall)[0]
    return float(fpr[idx[0]]) if len(idx) else float("nan")


def make_model(name, seed, params=None):
    # tuned params must OVERRIDE the defaults, not collide with them: passing e.g. a tuned
    # n_estimators alongside a hard-coded one raises TypeError, which the HPO fitness then
    # silently swallows as "invalid config" — so merge dicts instead of stacking kwargs.
    params = params or {}
    if name == "LogReg":
        kw = {"max_iter": 1000, "class_weight": "balanced"} | params
        return LogisticRegression(**kw)
    if name == "RandomForest":
        kw = {"n_estimators": 300, "class_weight": "balanced",
              "n_jobs": -1, "random_state": seed} | params
        return RandomForestClassifier(**kw)
    if name == "HistGB":
        kw = {"class_weight": "balanced", "random_state": seed} | params
        return HistGradientBoostingClassifier(**kw)
    if name == "MLP":  # scaled inputs matter for a neural net on tabular features
        pipe = make_pipeline(StandardScaler(),
                             MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400,
                                           random_state=seed))
        return pipe.set_params(**params) if params else pipe  # keys use mlpclassifier__ prefix
    raise ValueError(f"unknown model: {name}")


def _metrics(y, score, thr=0.5):
    pred = (np.asarray(score) >= thr).astype(int)
    y = np.asarray(y)
    auc = roc_auc_score(y, score) if len(set(y)) > 1 else float("nan")
    return {"F1": f1_score(y, pred, zero_division=0),
            "PR-AUC": average_precision_score(y, score),
            "ROC-AUC": auc,
            "FPR@R0.90": fpr_at_recall(y, score)}


# every family is fitted unweighted; step 6 switches this off for the legacy arm
UNWEIGHTED = True


FAMILIES = ["LogReg", "RandomForest", "HistGB", "MLP", "XGBoost", "LightGBM", "CatBoost"]


WEIGHTED = {"LogReg", "RandomForest", "HistGB"}


def weighting_params(name, unweighted=None):
    u = UNWEIGHTED if unweighted is None else unweighted
    return {"class_weight": None} if (u and name in WEIGHTED) else None


def make_any_model(name: str, seed: int, params: dict | None = None):
    p = params or {}
    if name == "Stack[CB+LR]":
        from catboost import CatBoostClassifier
        from sklearn.ensemble import StackingClassifier
        from sklearn.linear_model import LogisticRegression
        return StackingClassifier(
            estimators=[("CB", CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1,
                                                  random_seed=seed, verbose=False)),
                        ("LR", LogisticRegression(max_iter=1000, class_weight="balanced"))],
            final_estimator=LogisticRegression(max_iter=1000),
            stack_method="predict_proba", cv=5, n_jobs=-1)
    if name == "XGBoost":
        from xgboost import XGBClassifier
        return XGBClassifier(**{"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
                                "eval_metric": "logloss", "random_state": seed, "n_jobs": -1,
                                **p})
    if name == "LightGBM":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(**{"n_estimators": 300, "num_leaves": 63, "learning_rate": 0.1,
                                 "random_state": seed, "n_jobs": -1, "verbosity": -1, **p})
    if name == "CatBoost":
        from catboost import CatBoostClassifier
        return CatBoostClassifier(**{"iterations": 300, "depth": 6, "learning_rate": 0.1,
                                     "random_seed": seed, "verbose": False, **p})
    return make_model(name, seed, params)


PR_GRID = np.linspace(0.02, 1.0, 200)


def pr_curve_row(y, score):
    from sklearn.metrics import precision_recall_curve
    prec, rec, _ = precision_recall_curve(y, score)
    return np.interp(PR_GRID, rec[::-1], prec[::-1])


def write_curves(curves, path):
    if not curves or not path:
        return
    import collections
    acc = collections.defaultdict(list)
    for c in curves:
        acc[(c["family"], c["protocol"])].append(c["precision"])
    rows = []
    for (fam, proto), mats in acc.items():
        mean = np.mean(mats, axis=0)
        rows += [{"family": fam, "protocol": proto, "n_seeds": len(mats),
                  "recall": round(float(r), 4), "precision": round(float(pv), 6)}
                 for r, pv in zip(PR_GRID, mean)]
    new = pd.DataFrame(rows)
    if os.path.exists(path):
        # a partial run (--families) refreshes only its own curves and keeps everyone else's
        old = pd.read_csv(path)
        keep = ~old.set_index(["family", "protocol"]).index.isin(list(acc))
        new = pd.concat([old[keep], new], ignore_index=True)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    new.to_csv(path, index=False)
    print(f"[+] {len(acc)} curve(s) -> {path}")


def run_one(name, Xtr, ytr, Xte, yte, seed, return_scores=False, unweighted=None):
    t0 = time.time()
    params = weighting_params(name, unweighted)
    m = make_any_model(name, seed, params)
    m.fit(Xtr, ytr)
    fit_s = time.time() - t0
    score = m.predict_proba(Xte)[:, 1]
    met = _metrics(yte, score)
    met.update({"family": name, "seed": seed, "fit_seconds": round(fit_s, 2)})
    return (met, np.asarray(yte), score) if return_scores else met


URL_CSV = "data/processed/dataset_url.csv"
ALIGNED = "data/processed/vn_compphish.csv"


def parse_date(s):
    s = (s if isinstance(s, str) else "").strip()[:10]
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    return None


def load():
    meta = pd.read_csv(URL_CSV, usecols=["url_norm", "label", "collected_at"], dtype=str)
    meta["date"] = meta["collected_at"].map(parse_date)
    df = add_label(pd.read_csv(ALIGNED))
    df = df.merge(meta.rename(columns={"url_norm": "url"})[["url", "date"]],
                  on="url", how="left")
    df = df.dropna(subset=[c for c in COMPPHISH if c in df.columns])
    df["rdom"] = df["dom"].astype(str).str.lower().map(registered_domain)
    return df


def split_phishing(ph: pd.DataFrame, cut: float, test_after: str | None = None):
    if test_after:
        d = pd.Timestamp(test_after)
        ph_tr, ph_te = ph[ph.date <= d], ph[ph.date > d]
    else:
        k = int(len(ph) * cut)
        ph_tr, ph_te = ph.iloc[:k], ph.iloc[k:]
    leaked = ph_te.rdom.isin(set(ph_tr.rdom))
    return ph_tr, ph_te[~leaked], int(leaked.sum())


def benign_mask(be: pd.DataFrame, rng, cut: float, grouped: bool = False) -> np.ndarray:
    if not grouped:
        return rng.rand(len(be)) < cut
    doms = be["rdom"].astype(str).to_numpy()
    uniq = np.unique(doms)
    return np.isin(doms, uniq[rng.rand(len(uniq)) < cut])


def match_prior(te: pd.DataFrame, prior: float, rng) -> pd.DataFrame:
    ph, be = te[te.y == 1], te[te.y == 0]
    k = int(round(len(ph) * (1 - prior) / prior))
    if k >= len(be):
        return te
    return pd.concat([ph, be.iloc[np.sort(rng.choice(len(be), k, replace=False))]])


DEFAULT_TEST_FRAC = 0.30


def corrected_paired_t(d, test_frac: float = DEFAULT_TEST_FRAC) -> dict:
    d = np.asarray(d, dtype=float)
    k = len(d)
    mean = float(np.mean(d))
    var = float(np.var(d, ddof=1)) if k > 1 else 0.0
    out = {"k": k, "mean": mean, "wins": int(np.sum(d > 0)),
           "p": 1.0, "p_naive": 1.0, "se": 0.0, "inflation": float("nan")}
    if k < 2 or var <= 0:
        return out
    try:
        from scipy import stats
    except ImportError:  # keep the pipeline runnable without scipy; verdicts fall back to "no evidence"
        return out
    factor = 1.0 / k + test_frac / (1.0 - test_frac)
    se = math.sqrt(factor * var)
    se_naive = math.sqrt(var / k)
    t = mean / se
    out.update(p=float(2 * stats.t.sf(abs(t), k - 1)),
               p_naive=float(2 * stats.t.sf(abs(mean / se_naive), k - 1)),
               se=se, t=float(t), inflation=se / se_naive)
    return out


def bh_adjust(pvals, alpha=0.05):
    idx = sorted(range(len(pvals)), key=lambda i: pvals[i])
    m = len(pvals)
    adj = [1.0] * m
    running = 1.0
    for rank in range(m, 0, -1):          # walk up from the largest p-value
        i = idx[rank - 1]
        running = min(running, pvals[i] * m / rank)
        adj[i] = min(1.0, running)
    return adj, [a <= alpha for a in adj]


df_all = load()
print(f"{len(df_all):,} URLs, {int(df_all.y.sum()):,} phishing; dated phishing: {int(((df_all.y == 1) & df_all.date.notna()).sum()):,}")

## Bước 2: Benchmark bảy họ

Hai giao thức trên toàn bộ 53.116 URL: `random` chia ngẫu nhiên phân tầng 70/30, lặp 5 seed (cách đánh giá được công bố nhiều nhất); `temporal` dùng cột
`split` có sẵn trong bộ dữ liệu (cách chia chính thức của bộ PhishVN). Hồi quy logistic không phụ thuộc seed nên giao thức `temporal` chỉ chạy một lần.

In [ ]:
def run_benchmark(**overrides):
    args = SimpleNamespace(**{**{'inp': 'data/processed/vn_compphish.csv', 'families': None, 'seeds': 5, 'out': 'data/processed/p2/p2_benchmark.csv', 'curves': 'data/processed/p2/p2_pr_curves.csv', 'unweighted': True}, **overrides})
    if args.families is None:
        args.families = FAMILIES
    df = add_label(pd.read_csv(args.inp))
    feats = [c for c in COMPPHISH if c in df.columns]
    if len(feats) != len(COMPPHISH):
        raise SystemExit(f"corpus lacks CompPhish schema ({len(feats)}/{len(COMPPHISH)} features)")
    df = df.dropna(subset=feats)
    protocols = {}
    X, y = df[feats].to_numpy(float), df["y"].to_numpy(int)
    protocols["random"] = lambda s: train_test_split(X, y, test_size=0.30, stratify=y,
                                                     random_state=s)
    if "split" in df.columns:
        tr, te = df[df.split == "train"], df[df.split == "test"]
        Xtr, ytr = tr[feats].to_numpy(float), tr["y"].to_numpy(int)
        Xte, yte = te[feats].to_numpy(float), te["y"].to_numpy(int)
        protocols["temporal"] = lambda s: (Xtr, Xte, ytr, yte)
        print(f"temporal split: train={len(tr)} test={len(te)}")
    else:
        print("[!] no `split` column — temporal protocol skipped")
    print(f"corpus: {len(df)} rows, {int(y.sum())} phishing / {len(y) - int(y.sum())} benign")
    rows, curves = [], []
    for name in args.families:
        for proto, splitter in protocols.items():
            seeds = [0] if (name in DETERMINISTIC and proto == "temporal") else range(args.seeds)
            for s in seeds:
                Xtr, Xte, ytr, yte = splitter(s)
                met, y_t, sc = run_one(name, Xtr, ytr, Xte, yte, s, return_scores=True,
                                       unweighted=args.unweighted)
                met["protocol"] = proto
                rows.append(met)
                curves.append({"family": name, "protocol": proto, "seed": s,
                               "precision": pr_curve_row(y_t, sc)})
                print(f"  {name:<13} {proto:<9} seed={s} "
                      f"F1={met['F1']:.3f} PR-AUC={met['PR-AUC']:.3f} "
                      f"FPR@R0.90={met['FPR@R0.90']:.3f} ({met['fit_seconds']}s)")
    out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    out.to_csv(args.out, index=False)
    write_curves(curves, args.curves)
    print(f"\n[+] {len(out)} runs -> {args.out}")
    agg = (out.groupby(["family", "protocol"])[["F1", "PR-AUC", "ROC-AUC", "FPR@R0.90"]]
              .agg(["mean", "std"]).round(4))
    print(agg.to_string())


run_benchmark()
check_table("2 benchmark", "p2_benchmark.csv", MET_TOL)
check_table("2 benchmark", "p2_pr_curves.csv", CURVE_TOL)
report("2 benchmark")

## Bước 3: Chia theo thời gian nghiêm ngặt

Chỉ dùng URL lừa đảo có ngày. Giao thức `temporal_strict`: 70% URL lừa đảo sớm nhất làm train, phần sau làm test, có guard tên miền (Bước 1); lớp benign chia
ngẫu nhiên 70/30 theo seed. Đối chứng `random_same_rows`: cùng các dòng đó nhưng URL lừa đảo được chia ngẫu nhiên, với cùng tỷ lệ. Hiệu giữa hai giao thức là phần
do thời gian gây ra, khi tập dòng được giữ nguyên.

In [ ]:
def run_strict(**overrides):
    args = SimpleNamespace(**{**{'families': None, 'seeds': 5, 'cut': 0.7, 'test_after': None, 'out': 'data/processed/p2/p2_temporal_strict.csv', 'curves': 'data/processed/p2/p2_pr_curves_strict.csv', 'guard_control': False, 'benign_grouped': False, 'match_prior': False}, **overrides})
    if args.families is None:
        args.families = FAMILIES
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, n_leaked = split_phishing(ph, args.cut, args.test_after)
    if ph_te.empty:
        raise SystemExit(f"no dated phishing after {args.test_after}: the refresh window has "
                         f"not arrived (corpus max date {ph.date.max().date()})")
    print(f"phishing dated: {len(ph)}  train<= {ph_tr.date.max().date()} ({len(ph_tr)})  "
          f"test>= {ph_te.date.min().date()} ({len(ph_te)}; {n_leaked} test rows dropped as "
          f"same-registrable-domain re-detections)")
    print(f"benign pool: {len(be)} ({'grouped by registrable domain' if args.benign_grouped else 'random'}"
          f" {args.cut:.2f} per seed)")
    protos = ("temporal_strict", "random_same_rows_guarded") if args.guard_control \
        else ("temporal_strict", "random_same_rows")
    rows, curves = [], []
    for name in args.families:
        for proto in protos:
            # even "deterministic" families need all seeds here: the benign 70/30 mask is drawn
            # per seed, so no family is deterministic under either protocol
            for s in range(args.seeds):
                rng = np.random.RandomState(s)
                bmask = benign_mask(be, rng, args.cut, args.benign_grouped)
                if proto == "temporal_strict":
                    tr = pd.concat([ph_tr, be[bmask]])
                    te = pd.concat([ph_te, be[~bmask]])
                else:
                    pool = pd.concat([ph_tr, ph_te])
                    pmask = rng.rand(len(pool)) < (len(ph_tr) / (len(ph_tr) + len(ph_te)))
                    p_tr, p_te = pool[pmask], pool[~pmask]
                    if proto == "random_same_rows_guarded":
                        # The temporal arm's guard, applied to the random control: drop from TEST
                        # any row whose registrable domain occurs in TRAIN. Same direction, same
                        # granularity, same PSL logic -- so the only remaining difference between
                        # the two arms is how rows were assigned.
                        p_te = p_te[~p_te.rdom.isin(set(p_tr.rdom))]
                    tr = pd.concat([p_tr, be[bmask]])
                    te = pd.concat([p_te, be[~bmask]])
                    if args.match_prior:
                        te = match_prior(te, len(ph_te) / (len(ph_te) + int((~bmask).sum())), rng)
                met, y_t, sc = run_one(name, tr[feats].to_numpy(float),
                                       tr["y"].to_numpy(int), te[feats].to_numpy(float),
                                       te["y"].to_numpy(int), s, return_scores=True)
                met["protocol"] = proto
                rows.append(met)
                curves.append({"family": name, "protocol": proto, "seed": s,
                               "precision": pr_curve_row(y_t, sc)})
                print(f"  {name:<13} {proto:<17} seed={s} F1={met['F1']:.3f} "
                      f"PR-AUC={met['PR-AUC']:.3f} FPR@R0.90={met['FPR@R0.90']:.3f}")
    out = pd.DataFrame(rows)
    out.to_csv(args.out, index=False)
    write_curves(curves, args.curves)
    print(f"\n[+] {len(out)} runs -> {args.out}")
    print(out.groupby(["family", "protocol"])[["F1", "PR-AUC", "ROC-AUC", "FPR@R0.90"]]
             .mean().round(4).to_string())


run_strict()
check_table("3 temporal strict", "p2_temporal_strict.csv", MET_TOL)
check_table("3 temporal strict", "p2_pr_curves_strict.csv", CURVE_TOL)
report("3 temporal strict")

## Bước 4: Các đối chứng của giao thức thời gian

Giao thức thời gian có guard tên miền ở phía lừa đảo, còn đối chứng ngẫu nhiên ở Bước 3 thì không, nên một phần chênh lệch giữa hai bên là do guard. Bốn biến
thể đo phần đó:
- `guarded`: đối chứng ngẫu nhiên cũng có guard (`random_same_rows_guarded`), để giao thức là biến duy nhất;
- `guarded_pm`: thêm việc lấy mẫu bớt benign trong test của đối chứng cho tỷ lệ lớp bằng bên thời gian;
- `guarded_pm_bgroup`: như trên, và lớp benign được chia theo tên miền đăng ký (mỗi tên miền nằm trọn một phía);
- `bgroup`: giao thức gốc của Bước 3 với benign chia theo tên miền.

In [ ]:
for out, kw in (("p2_temporal_strict_guarded.csv", dict(guard_control=True)),
                ("p2_temporal_strict_guarded_pm.csv", dict(guard_control=True, match_prior=True)),
                ("p2_temporal_strict_guarded_pm_bgroup.csv", dict(guard_control=True, match_prior=True, benign_grouped=True)),
                ("p2_temporal_strict_bgroup.csv", dict(benign_grouped=True))):
    run_strict(out=f"{RES}/{out}", curves="", **kw)
    check_table("4 guard controls", out, MET_TOL)
report("4 guard controls")

## Bước 5: CatBoost với 20 seed

Giao thức của Bước 3, chỉ CatBoost, 20 seed: dùng cho phép so có cặp với stacking ở Bước 16, cần nhiều lần chia hơn năm.

In [ ]:
run_strict(families=["CatBoost"], seeds=20, out=f"{RES}/p2_temporal_strict_cb_k20.csv", curves="")
check_table("5 CatBoost k20", "p2_temporal_strict_cb_k20.csv", MET_TOL)
report("5 CatBoost k20")

## Bước 6: Cấu hình cũ có trọng số lớp

Trước bản sửa, hồi quy logistic, random forest và HistGB được huấn luyện với `class_weight="balanced"`, còn bốn họ kia thì không. Bước này chạy lại Bước 2 và 3
theo cấu hình trộn đó, chỉ để bài đối chiếu: riêng việc cân trọng số đã tạo ra cả khoảng cách 5,3 điểm F1 giữa các họ mà bản đầu của bài từng nêu.

In [ ]:
UNWEIGHTED = False
try:
    run_benchmark(out=f"{RES}/p2_benchmark_mixedweights.csv", curves="", unweighted=False)
    run_strict(out=f"{RES}/p2_temporal_strict_mixedweights.csv", curves="")
finally:
    UNWEIGHTED = True
check_table("6 legacy weighting", "p2_benchmark_mixedweights.csv", MET_TOL)
check_table("6 legacy weighting", "p2_temporal_strict_mixedweights.csv", MET_TOL)
report("6 legacy weighting")

## Bước 7: Tỷ lệ lớp hay tập dòng

Khi chuyển từ toàn bộ corpus sang các dòng có ngày, hai thứ đổi cùng lúc: tỷ lệ lừa đảo (0,69 xuống 0,55) và chính tập URL lừa đảo. Bốn arm tách hai thứ đó bằng
lấy mẫu lại, mỗi arm chia ngẫu nhiên phân tầng: toàn bộ ở tỷ lệ gốc; toàn bộ với lớp lừa đảo rút bớt về đúng số dòng có ngày; các dòng có ngày ở tỷ lệ của chúng;
và các dòng có ngày với benign rút bớt về tỷ lệ của toàn bộ. Ngoài F1 tại 0,5, mỗi lần chạy còn ghi F1 lớn nhất trên đường PR (`maxF1`).

Bảng của bài được ghi trước khi mã thêm hai cột đếm `n_phish` và `n_benign`, nên phép so dùng các cột chung của hai bảng.

Bảng của bài cũng được chạy với một luồng BLAS. MLP cộng các phép nhân ma trận theo thứ tự phụ thuộc số luồng, nên chạy đa luồng thì các dòng MLP lệch ở chữ số
thứ ba (tới khoảng 0,004); các họ cây không bị ảnh hưởng. Bước này vì vậy chạy dưới `threadpool_limits(1, "blas")`, chậm hơn nhưng khớp đúng.

In [ ]:
ARMS = ("full_natural", "full_matched", "dated_natural", "dated_matched")


def prior_max_f1(y, score) -> float:
    p, r, _ = precision_recall_curve(y, score)
    with np.errstate(divide="ignore", invalid="ignore"):
        f = np.where(p + r > 0, 2 * p * r / (p + r), 0.0)
    return float(np.nanmax(f))


def draw(arm: str, ph_all: pd.DataFrame, ph_dated: pd.DataFrame, be: pd.DataFrame,
         rng: np.random.RandomState) -> pd.DataFrame:
    n_dated, n_be = len(ph_dated), len(be)
    prior_full = len(ph_all) / (len(ph_all) + n_be)
    if arm == "full_natural":
        return pd.concat([ph_all, be])
    if arm == "full_matched":
        idx = rng.choice(len(ph_all), n_dated, replace=False)
        return pd.concat([ph_all.iloc[np.sort(idx)], be])
    if arm == "dated_natural":
        return pd.concat([ph_dated, be])
    if arm == "dated_matched":
        n_keep = int(round(n_dated * (1 - prior_full) / prior_full))
        idx = rng.choice(n_be, n_keep, replace=False)
        return pd.concat([ph_dated, be.iloc[np.sort(idx)]])
    raise ValueError(arm)


def run_prior_control(**overrides):
    a = SimpleNamespace(**{**{'families': None, 'seeds': 5, 'out': 'data/processed/p2/p2_prior_control.csv'}, **overrides})
    if a.families is None:
        a.families = FAMILIES
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph_all = df[df.y == 1].reset_index(drop=True)
    ph_dated = ph_all[ph_all.date.notna()].reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    print(f"phishing {len(ph_all)} (dated {len(ph_dated)}), benign {len(be)}; "
          f"prior full {len(ph_all)/(len(ph_all)+len(be)):.3f}, "
          f"dated {len(ph_dated)/(len(ph_dated)+len(be)):.3f}")
    rows = []
    for arm in ARMS:
        for s in range(a.seeds):
            pool = draw(arm, ph_all, ph_dated, be, np.random.RandomState(1000 + s))
            X, y = pool[feats].to_numpy(float), pool["y"].to_numpy(int)
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y,
                                                  random_state=s)
            for name in a.families:
                met, y_t, sc = run_one(name, Xtr, ytr, Xte, yte, s, return_scores=True)
                met.update(arm=arm, prior=round(float(y.mean()), 4), n_rows=len(pool),
                           n_phish=int(y.sum()), n_benign=int(len(y) - y.sum()),
                           n_test=len(yte), maxF1=prior_max_f1(y_t, sc))
                rows.append(met)
                print(f"  {arm:<14} seed={s} {name:<13} F1={met['F1']:.3f} "
                      f"maxF1={met['maxF1']:.3f} PR-AUC={met['PR-AUC']:.3f}", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)
    out = pd.DataFrame(rows)
    out.to_csv(a.out, index=False)
    print(f"\n[+] {len(out)} runs -> {a.out}")
    print(out.groupby(["arm", "family"])[["F1", "maxF1", "PR-AUC"]].mean().round(4)
             .unstack("arm").to_string())


with threadpool_limits(1, "blas"):  # the paper's table was written single-threaded; MLP sums depend on it
    run_prior_control()
check_table("7 prior or rows", "p2_prior_control.csv", MET_TOL, shared_columns=True)
report("7 prior or rows")

## Bước 8: Ngưỡng chọn trên validation

F1 tại ngưỡng cố định 0,5 phạt những mô hình có điểm số lệch thang. Bước này tách 10% tập train làm validation (theo thời gian với thiết kế thời gian), chọn
ngưỡng tối đa hoá F1 trên đó, rồi đo F1 trên test tại ngưỡng đó (`F1_val`), cạnh F1 tại 0,5 và F1 lớn nhất có thể trên chính test (`F1_oracle`).

Ngưỡng chọn là điểm cực đại trên một đường cong bậc thang, nên một thay đổi cỡ 10⁻¹⁶ trong xác suất của random forest (do cộng song song) có thể đẩy nó sang
điểm kế bên. Vì vậy hai cột phụ thuộc ngưỡng đó (`tau`, `F1_val`) được so với dung sai rộng hơn, `THR_TOL` = 10⁻³; các cột khác giữ `MET_TOL`.

In [ ]:
DESIGNS = ("random_full", "random_same_rows", "temporal_strict")


VAL_FRAC = 0.10


def best_threshold(y, score) -> float:
    p, r, thr = precision_recall_curve(y, score)
    with np.errstate(divide="ignore", invalid="ignore"):
        f = np.where(p + r > 0, 2 * p * r / (p + r), 0.0)[:-1]
    return float(thr[int(np.nanargmax(f))])


def carve(tr: pd.DataFrame, design: str, seed: int):
    rng = np.random.RandomState(10_000 + seed)
    if design == "temporal_strict":
        ph = tr[tr.y == 1].sort_values("date")
        be = tr[tr.y == 0]
        k = int(len(ph) * (1 - VAL_FRAC))
        bm = rng.rand(len(be)) < VAL_FRAC
        fit = pd.concat([ph.iloc[:k], be[~bm]])
        val = pd.concat([ph.iloc[k:], be[bm]])
        return fit, val
    fit, val = train_test_split(tr, test_size=VAL_FRAC, stratify=tr.y, random_state=seed)
    return fit, val


def run_val_threshold(**overrides):
    a = SimpleNamespace(**{**{'families': None, 'seeds': 5, 'cut': 0.7, 'out': 'data/processed/p2/p2_val_threshold.csv'}, **overrides})
    if a.families is None:
        a.families = FAMILIES
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, _ = split_phishing(ph, a.cut)
    rows = []
    for design in DESIGNS:
        for s in range(a.seeds):
            if design == "random_full":
                tr, te = train_test_split(df, test_size=0.30, stratify=df.y, random_state=s)
            else:
                rng = np.random.RandomState(s)          # the canonical per-seed benign mask
                bmask = rng.rand(len(be)) < a.cut
                if design == "temporal_strict":
                    tr = pd.concat([ph_tr, be[bmask]])
                    te = pd.concat([ph_te, be[~bmask]])
                else:
                    pool = pd.concat([ph_tr, ph_te])
                    pmask = rng.rand(len(pool)) < (len(ph_tr) / (len(ph_tr) + len(ph_te)))
                    tr = pd.concat([pool[pmask], be[bmask]])
                    te = pd.concat([pool[~pmask], be[~bmask]])
            fit, val = carve(tr, design, s)
            for name in a.families:
                m = make_any_model(name, s, weighting_params(name))
                m.fit(fit[feats].to_numpy(float), fit.y.to_numpy(int))
                sv = m.predict_proba(val[feats].to_numpy(float))[:, 1]
                tau = best_threshold(val.y.to_numpy(int), sv)
                st = m.predict_proba(te[feats].to_numpy(float))[:, 1]
                yte = te.y.to_numpy(int)
                met = _metrics(yte, st)                      # F1 here is at tau = 0.5
                met.update(family=name, seed=s, protocol=design, tau=round(tau, 4),
                           F1_val=f1_score(yte, (st >= tau).astype(int)),
                           F1_oracle=prior_max_f1(yte, st),
                           n_fit=len(fit), n_val=len(val), n_test=len(te),
                           val_last_date=str(val[val.y == 1].date.max().date())
                           if design == "temporal_strict" else "")
                rows.append(met)
                print(f"  {design:<17} seed={s} {name:<13} tau*={tau:.3f} "
                      f"F1@0.5={met['F1']:.3f} F1@tau*={met['F1_val']:.3f} "
                      f"oracle={met['F1_oracle']:.3f}", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)
    out = pd.DataFrame(rows)
    out.to_csv(a.out, index=False)
    print(f"\n[+] {len(out)} runs -> {a.out}")
    print(out.groupby(["protocol", "family"])[["F1", "F1_val", "F1_oracle"]].mean().round(4)
             .to_string())


run_val_threshold()
THR_TOL = 1e-3     # columns read at a threshold chosen on the validation rows
check_table("8 validation threshold", "p2_val_threshold.csv", MET_TOL, col_tol={"tau": THR_TOL, "F1_val": THR_TOL})
report("8 validation threshold")

## Bước 9: Chia theo thời gian cho cả hai lớp

Bước 3 chỉ chia lớp lừa đảo theo thời gian. Ở đây mọi dòng có ngày, cả benign, được cắt tại cùng một ngày (ngày cắt của lớp lừa đảo), có guard tên miền ở cả hai
lớp; đối chứng chia ngẫu nhiên với cùng tỷ lệ train của từng lớp và cùng guard. Hồi quy logistic ở bước này nhạy với cách thư viện BLAS chia luồng, nên bước chạy
với BLAS một luồng, đúng như kết quả của bài.

In [ ]:
def guard(te: pd.DataFrame, tr: pd.DataFrame) -> pd.DataFrame:
    return te[~te.rdom.isin(set(tr.rdom))]


def run_twosided(**overrides):
    a = SimpleNamespace(**{**{'families': None, 'seeds': 5, 'cut': 0.7, 'out': 'data/processed/p2/p2_temporal_twosided.csv'}, **overrides})
    if a.families is None:
        a.families = FAMILIES
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    dated = df[df.date.notna()].sort_values("date").reset_index(drop=True)
    ph = dated[dated.y == 1]
    be = dated[dated.y == 0]
    cut_date = ph.date.iloc[int(len(ph) * a.cut) - 1]      # the canonical phishing cut
    tr_all = dated[dated.date <= cut_date]
    te_all = guard(dated[dated.date > cut_date], tr_all)
    n_dropped = int((dated.date > cut_date).sum()) - len(te_all)
    print(f"cut {cut_date.date()}: train {len(tr_all)} (phish {int(tr_all.y.sum())}, benign "
          f"{int((tr_all.y == 0).sum())}), test {len(te_all)} (phish {int(te_all.y.sum())}, benign "
          f"{int((te_all.y == 0).sum())}); {n_dropped} test rows dropped by the domain guard")
    rate = {1: (tr_all.y == 1).sum() / len(ph), 0: (tr_all.y == 0).sum() / len(be)}
    rows = []
    for name in a.families:
        for proto in ("temporal_both", "random_both_guarded"):
            for s in range(a.seeds):
                if proto == "temporal_both":
                    tr, te = tr_all, te_all
                else:
                    rng = np.random.RandomState(s)
                    mask = np.array([rng.rand() < rate[int(y)] for y in dated.y])
                    tr, te = dated[mask], guard(dated[~mask], dated[mask])
                met, _, _ = run_one(name, tr[feats].to_numpy(float), tr.y.to_numpy(int),
                                    te[feats].to_numpy(float), te.y.to_numpy(int), s,
                                    return_scores=True)
                met.update(protocol=proto, n_train=len(tr), n_test=len(te),
                           n_test_benign=int((te.y == 0).sum()), cut_date=str(cut_date.date()))
                rows.append(met)
                print(f"  {name:<13} {proto:<20} seed={s} F1={met['F1']:.3f} "
                      f"ROC={met['ROC-AUC']:.3f} PR={met['PR-AUC']:.3f} "
                      f"FPR@0.90={met['FPR@R0.90']:.3f}", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)
    out = pd.DataFrame(rows)
    out.to_csv(a.out, index=False)
    print(f"\n[+] {len(out)} runs -> {a.out}")
    print(out.groupby(["family", "protocol"])[["F1", "PR-AUC", "ROC-AUC", "FPR@R0.90"]]
             .mean().round(4).to_string())


with threadpool_limits(1, "blas"):
    run_twosided()
check_table("9 two-sided temporal", "p2_temporal_twosided.csv", MET_TOL)
report("9 two-sided temporal")

## Bước 10: Dòng trùng và rò rỉ

Không cần mô hình. Với chia ngẫu nhiên trên toàn bộ corpus, bước này đếm tỷ lệ dòng test có vector đặc trưng giống hệt một dòng train (`test_twin_rate`) và tỷ lệ
có cùng tên miền đăng ký với một dòng train (`test_regdom_shared_rate`); F1 của một "bộ phân loại tra bảng" chỉ nhớ nhãn đa số của từng vector đặc trưng; F1 của
việc đoán mọi thứ là lừa đảo; và trần F1 mà lược đồ đặc trưng cho phép (khi nhiều URL khác nhãn có cùng vector). Sau đó làm lại các tỷ lệ trùng cho hai thiết kế
trên các dòng có ngày.

In [ ]:
OUT_PROTO = "data/processed/p2/p2_dup_leakage_protocols.csv"


def _group_labels(keys, labels):
    acc = collections.defaultdict(list)
    for k, lab in zip(keys, labels):
        acc[k].append(int(lab))
    return acc


def _rates(fv, rdom, itr, ite):
    tr_fv, tr_rd = set(fv[itr]), set(rdom[itr])
    return (float(np.mean([k in tr_fv for k in fv[ite]])),
            float(np.mean([r in tr_rd for r in rdom[ite]])))


def protocol_rates(seeds: int, out_path: str):
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    cut = int(len(ph) * 0.70)
    ph_tr, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr.rdom))]

    def keys(frame):
        return (np.array([hash(t) for t in map(tuple, frame[feats].to_numpy(float))]),
                frame["rdom"].to_numpy())

    rows = []
    for s in range(seeds):
        rng = np.random.RandomState(s)
        bmask = rng.rand(len(be)) < 0.70
        pool = pd.concat([ph_tr, ph_te])
        pmask = rng.rand(len(pool)) < (len(ph_tr) / (len(ph_tr) + len(ph_te)))
        for proto, tr, te in (
                ("temporal_strict", pd.concat([ph_tr, be[bmask]]),
                 pd.concat([ph_te, be[~bmask]])),
                ("random_same_rows", pd.concat([pool[pmask], be[bmask]]),
                 pd.concat([pool[~pmask], be[~bmask]]))):
            fv = np.concatenate([keys(tr)[0], keys(te)[0]])
            rd = np.concatenate([keys(tr)[1], keys(te)[1]])
            itr = np.arange(len(tr))
            ite = np.arange(len(tr), len(tr) + len(te))
            twin, dom = _rates(fv, rd, itr, ite)
            # the phishing side alone is where the guard operates and where drift lives
            pv_tr = keys(tr[tr.y == 1])
            pv_te = keys(te[te.y == 1])
            fvp = np.concatenate([pv_tr[0], pv_te[0]])
            rdp = np.concatenate([pv_tr[1], pv_te[1]])
            twin_p, dom_p = _rates(fvp, rdp, np.arange(len(pv_tr[0])),
                                   np.arange(len(pv_tr[0]), len(fvp)))
            rows.append({"protocol": proto, "seed": s,
                         "test_twin_rate": round(twin, 4),
                         "test_regdom_shared_rate": round(dom, 4),
                         "phish_test_twin_rate": round(twin_p, 4),
                         "phish_test_regdom_shared_rate": round(dom_p, 4)})
            print(f"  {proto:<17} seed={s} twin={twin:.4f} regdom={dom:.4f} "
                  f"(phishing side {twin_p:.4f}/{dom_p:.4f})")
    out = pd.DataFrame(rows)
    out.to_csv(out_path, index=False)
    print(f"[+] {len(out)} rows -> {out_path}")
    return out


def run_dup_leakage(**overrides):
    args = SimpleNamespace(**{**{'inp': 'data/processed/vn_compphish.csv', 'seeds': 5, 'out': 'data/processed/p2/p2_dup_leakage.csv'}, **overrides})
    df = add_label(pd.read_csv(args.inp)).dropna(subset=COMPPHISH)
    X = df[COMPPHISH].to_numpy(float)
    y = df["y"].to_numpy(int)
    # Exact float tuples, not rounded strings: rounding merges vectors the learner can still
    # tell apart and would overstate the leak.
    fv = np.array([hash(t) for t in map(tuple, X)])
    rdom = df["dom"].astype(str).str.lower().map(registered_domain).to_numpy()
    groups = _group_labels(fv, y)
    mixed = {k for k, s in groups.items() if len(set(s)) > 1}
    mixed_rows = int(np.isin(fv, list(mixed)).sum()) if mixed else 0
    # The schema ceiling: majority label per exact feature vector, fitted and scored on
    # everything. No split, no learner -- the best F1 these 21 features admit on this corpus.
    maj = {k: int(sum(g) * 2 >= len(g)) for k, g in groups.items()}
    ceiling = float(f1_score(y, np.array([maj[k] for k in fv])))
    census = {
        "rows": len(df),
        "unique_regdom": int(len(set(rdom))),
        "unique_featvec": int(len(groups)),
        "mixed_featvec_groups": len(mixed),
        "rows_in_mixed_groups": mixed_rows,
        "positive_rate": float(y.mean()),
        "schema_ceiling_f1": round(ceiling, 4),
    }
    print("census: " + "  ".join(f"{k}={v}" for k, v in census.items()))
    rows = []
    for s in range(args.seeds):
        itr, ite = train_test_split(np.arange(len(df)), test_size=0.30, stratify=y,
                                    random_state=s)
        twin, dom = _rates(fv, rdom, itr, ite)

        acc = _group_labels(fv[itr], y[itr])
        table = {k: int(sum(v) * 2 >= len(v)) for k, v in acc.items()}
        fallback = int(y[itr].mean() >= 0.5)
        pred = np.array([table.get(k, fallback) for k in fv[ite]])

        r = dict(census, seed=s,
                 test_twin_rate=round(twin, 4),
                 test_regdom_shared_rate=round(dom, 4),
                 lookup_oracle_f1=round(float(f1_score(y[ite], pred)), 4),
                 all_positive_f1=round(float(f1_score(y[ite], np.ones(len(ite), int))), 4))
        rows.append(r)
        print(f"  seed={s} twin={twin:.4f} regdom={dom:.4f} "
              f"oracle_F1={r['lookup_oracle_f1']:.4f} allpos_F1={r['all_positive_f1']:.4f}")
    out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    out.to_csv(args.out, index=False)
    print(f"\n[+] {len(out)} rows -> {args.out}")
    print(out[["test_twin_rate", "test_regdom_shared_rate", "lookup_oracle_f1",
               "all_positive_f1"]].mean().round(4).to_string())
    print("\ndated-row designs (run_p2_temporal_strict's own splits):")
    protocol_rates(args.seeds, OUT_PROTO)


run_dup_leakage()
check_table("10 duplicates", "p2_dup_leakage.csv", 0)
check_table("10 duplicates", "p2_dup_leakage_protocols.csv", 0)
report("10 duplicates")

## Bước 11: Ma trận chuyển giao giữa bốn corpus

Bốn corpus (PhishVN, PhiUSIIL, ISCX-URL2016, PhishStorm) đã ở cùng lược đồ 21 đặc trưng. Ô (i, j) huấn luyện trên corpus i và kiểm tra trên corpus j, trung bình
5 seed (`cell_score`). Ô đường chéo kiểm tra trong chính corpus: PhishVN dùng cách chia chính thức của nó; ba corpus kia dùng chia 70/30 theo nhóm tên miền
(`in_dataset_split`), vì ở các corpus đó rất nhiều dòng chung tên miền, và chia ngẫu nhiên từng dòng sẽ để cùng tên miền nằm cả hai phía. Khoảng cách tổng quát
hoá là trung bình đường chéo trừ trung bình ngoài đường chéo.

Bước này chạy 17 ma trận: random forest, hồi quy logistic và HistGB theo F1 và ROC-AUC; CatBoost theo F1; ba họ đó khi bỏ ba đặc trưng mà SHAP chỉ ra là
mang dấu vết riêng của từng corpus (`tld_len`, `subdom_cnt`, `dot_cnt`; arm "pruned"); và random forest, hồi quy logistic với CORAL, cách căn chỉnh phân phối
không giám sát: chuẩn hoá từng corpus rồi biến đổi đặc trưng nguồn cho ma trận hiệp phương sai của nó khớp với corpus đích (`CoralAligner`).

In [ ]:
LEGACY_SPLIT = False


def load_corpus(path):
    df = pd.read_csv(path)
    df = add_label(df)
    for c in COMPPHISH:
        if c not in df:
            df[c] = 0
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    if df["y"].nunique() < 2:
        raise SystemExit(f"{path}: needs both classes (phishing + benign).")
    return df


def in_dataset_split(df, seed):
    if "split" in df and df["split"].astype(str).isin(["train", "test"]).any():
        tr = df[df["split"] == "train"]
        te = df[df["split"] == "test"]
        if len(te) and te["y"].nunique() > 1:
            return tr, te
    if LEGACY_SPLIT or "dom" not in df:
        return train_test_split(df, test_size=0.3, stratify=df["y"], random_state=seed)
    gi, gj = next(GroupShuffleSplit(n_splits=1, test_size=0.3,
                                    random_state=seed).split(df, groups=df["dom"]))
    tr, te = df.iloc[gi], df.iloc[gj]
    if te["y"].nunique() < 2:  # degenerate draw: fall back rather than crash the cell
        return train_test_split(df, test_size=0.3, stratify=df["y"], random_state=seed)
    return tr, te


def _psd_power(C, p):
    w, V = np.linalg.eigh(C)
    return (V * np.clip(w, 0, None) ** p) @ V.T


class CoralAligner:

    def __init__(self, Xs, Xt, lam=1.0):
        Xs = np.asarray(Xs, float)
        Xt = np.asarray(Xt, float)
        self.mu_s, self.sd_s = Xs.mean(0), Xs.std(0) + 1e-9
        self.mu_t, self.sd_t = Xt.mean(0), Xt.std(0) + 1e-9
        Zs = (Xs - self.mu_s) / self.sd_s
        Zt = (Xt - self.mu_t) / self.sd_t
        d = Zs.shape[1]
        Cs = np.cov(Zs, rowvar=False) + lam * np.eye(d)
        Ct = np.cov(Zt, rowvar=False) + lam * np.eye(d)
        self.A = _psd_power(Cs, -0.5) @ _psd_power(Ct, 0.5)

    def source(self, X):
        return ((np.asarray(X, float) - self.mu_s) / self.sd_s) @ self.A

    def target(self, X):
        return (np.asarray(X, float) - self.mu_t) / self.sd_t


def _cell_metric(y, score, metric):
    if metric in ("MCC", "BalAcc"):
        from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef
        pred = (np.asarray(score) >= 0.5).astype(int)
        return (matthews_corrcoef(y, pred) if metric == "MCC"
                else balanced_accuracy_score(y, pred))
    return _metrics(y, score)[metric]


def cell_score(train_df, test_df, same, metric, seeds, model="RandomForest", tune=False,
               feats=COMPPHISH, adapt="none"):
    n_seed = 1 if model in DETERMINISTIC else seeds
    params = {}
    if tune:  # tune ONLY on the training corpus (never on the target test corpus)
        params = tune_params(model, train_df[feats], train_df["y"], 0)
    vals = []
    for s in range(n_seed):
        if same:
            tr, te = in_dataset_split(train_df, s)
        else:
            tr, te = train_df, test_df
        if model in MODEL_NAMES:
            # the baseline module's class_weight=balanced is dropped for the families that carry it.
            if UNWEIGHTED and model in {"LogReg", "RandomForest", "HistGB"}:
                params = {**params, "class_weight": None}
            m = make_model(model, s, params)
        else:
            m = make_any_model(model, s)
        if adapt == "coral" and not same:  # diagonal cells stay unadapted by construction
            al = CoralAligner(tr[feats].to_numpy(), te[feats].to_numpy())
            m = m.fit(al.source(tr[feats]), tr["y"])
            score = m.predict_proba(al.target(te[feats]))[:, 1]
        else:
            m = m.fit(tr[feats], tr["y"])
            score = m.predict_proba(te[feats])[:, 1]
        vals.append(_cell_metric(te["y"].to_numpy(), score, metric))
    return float(np.mean(vals)), float(np.std(vals))


def run_cross_dataset(**overrides):
    args = SimpleNamespace(**{**{'corpora': None, 'metric': 'F1', 'seeds': 5, 'model': 'RandomForest', 'tune': False, 'drop': '', 'adapt': 'none', 'legacy_split': False, 'out': ''}, **overrides})
    if args.corpora is None:
        args.corpora = CORPUS_ARGS
    drop = {f.strip() for f in args.drop.split(",") if f.strip()}
    unknown = drop - set(COMPPHISH)
    if unknown:
        raise SystemExit(f"--drop names not in the CompPhish schema: {sorted(unknown)}")
    feats = [c for c in COMPPHISH if c not in drop]
    if drop:
        print(f"[i] artefact-pruning: dropping {sorted(drop)} -> {len(feats)} features")
    corpora = {}
    for entry in args.corpora:
        if "=" not in entry:
            raise SystemExit(f"Bad --corpora entry (need name=path): {entry}")
        name, path = entry.split("=", 1)
        corpora[name] = load_corpus(path)
        print(f"[i] {name}: {len(corpora[name])} rows "
              f"(phishing={int(corpora[name]['y'].sum())})")
    names = list(corpora)
    mat = pd.DataFrame(index=names, columns=names, dtype=float)
    print(f"\nComputing {len(names)}x{len(names)} matrix "
          f"(metric={args.metric}, seeds={args.seeds}) ...")
    for i in names:
        for j in names:
            mean, std = cell_score(corpora[i], corpora[j], i == j, args.metric, args.seeds,
                                   model=args.model, tune=args.tune, feats=feats,
                                   adapt=args.adapt)
            mat.loc[i, j] = round(mean, 3)
            print(f"  train={i:14s} test={j:14s} {args.metric}={mean:.3f}±{std:.3f}"
                  f"{'  (in-dist)' if i == j else ''}")
    diag = np.array([mat.loc[n, n] for n in names], dtype=float)
    off = mat.values[~np.eye(len(names), dtype=bool)].astype(float)
    print("\n=== " + args.metric + " matrix (rows=train, cols=test) ===")
    print(mat.to_string())
    print(f"\nmean diagonal (in-distribution) = {diag.mean():.3f}")
    print(f"mean off-diagonal (transfer)    = {np.nanmean(off):.3f}")
    print(f"generalisation gap              = {diag.mean() - np.nanmean(off):.3f}  "
          f"(smaller = features transfer better)")
    if args.out:
        os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
        mat.to_csv(args.out)
        print(f"[+] saved matrix -> {args.out}")


CORPUS_ARGS = ["PhishVN=data/processed/vn_compphish.csv", "PhiUSIIL=data/processed/external/phiusiil_compphish.csv",
               "ISCXURL2016=data/processed/external/iscx_compphish.csv", "PhishStorm=data/processed/external/phishstorm_compphish.csv"]
PRUNE = "tld_len,subdom_cnt,dot_cnt"
MATRICES = []
for metric in ("F1", "ROC-AUC"):
    MATRICES += [(f"cross_dataset_{metric}.csv", dict(metric=metric)),
                 (f"cross_dataset_{metric}_LogReg.csv", dict(metric=metric, model="LogReg")),
                 (f"cross_dataset_{metric}_HistGB.csv", dict(metric=metric, model="HistGB")),
                 (f"cross_dataset_{metric}_pruned.csv", dict(metric=metric, drop=PRUNE)),
                 (f"cross_dataset_{metric}_LogReg_pruned.csv", dict(metric=metric, model="LogReg", drop=PRUNE)),
                 (f"cross_dataset_{metric}_HistGB_pruned.csv", dict(metric=metric, model="HistGB", drop=PRUNE)),
                 (f"cross_dataset_{metric}_coral.csv", dict(metric=metric, adapt="coral")),
                 (f"cross_dataset_{metric}_LogReg_coral.csv", dict(metric=metric, model="LogReg", adapt="coral"))]
MATRICES.append(("cross_dataset_F1_CatBoost.csv", dict(metric="F1", model="CatBoost")))
for name, kw in MATRICES:
    run_cross_dataset(out=f"{RES}/{name}", **kw)
    check_matrix("11 transfer matrices", name)
report("11 transfer matrices")

## Bước 12: Huấn luyện gộp

Liệu huấn luyện trên nhiều corpus cùng lúc có chuyển giao tốt hơn? Với mỗi corpus, random forest được huấn luyện trên hợp của ba corpus còn lại rồi kiểm tra trên
corpus bị giữ ra, với đủ 21 đặc trưng và với bộ đã bỏ ba đặc trưng, rồi so với ô tốt nhất và trung bình của các mô hình huấn luyện trên một corpus (đọc từ ma trận
của Bước 11).

In [ ]:
PROC = os.path.join(ROOT, "data/processed")


XD_CORPORA = {
    "PhishVN": "vn_compphish.csv",
    "PhiUSIIL": "external/phiusiil_compphish.csv",
    "ISCXURL2016": "external/iscx_compphish.csv",
    "PhishStorm": "external/phishstorm_compphish.csv",
}


XD_DROP = {"tld_len", "subdom_cnt", "dot_cnt"}


PRUNED = [c for c in COMPPHISH if c not in XD_DROP]


XD_SEEDS = 5


def combined_csv() -> str:
    return os.path.join(PROC, "p2", "combined_training.csv")


def run_combined_training(**overrides):
    args = SimpleNamespace(**{**{}, **overrides})
    data = {n: load_corpus(os.path.join(PROC, f)) for n, f in XD_CORPORA.items()}
    names = list(data)
    f1_mat = pd.read_csv(os.path.join(PROC, "p2", "cross_dataset_F1.csv"), index_col=0)
    roc_mat = pd.read_csv(os.path.join(PROC, "p2", "cross_dataset_ROC-AUC.csv"), index_col=0)
    rows = []
    for held in names:
        union = pd.concat([data[n] for n in names if n != held], ignore_index=True)
        te = data[held]
        print(f"held-out={held}: union train={len(union)} rows, test={len(te)}")
        for tag, feats in (("full", COMPPHISH), ("pruned", PRUNED)):
            t0 = time.time()
            f1s, rocs = [], []
            for s in range(XD_SEEDS):
                m = make_model("RandomForest", s, {"class_weight": None} if UNWEIGHTED else {}).fit(
                    union[feats], union["y"])
                score = m.predict_proba(te[feats])[:, 1]
                met = _metrics(te["y"].to_numpy(), score)
                f1s.append(met["F1"])
                rocs.append(met["ROC-AUC"])
            singles_f1 = [f1_mat.loc[i, held] for i in names if i != held]
            rows.append({
                "held_out": held, "config": tag,
                "F1": float(np.mean(f1s)), "F1_std": float(np.std(f1s)),
                "ROC-AUC": float(np.mean(rocs)), "ROC-AUC_std": float(np.std(rocs)),
                "best_single_F1": float(max(singles_f1)),
                "best_single_ROC": float(max(roc_mat.loc[i, held] for i in names if i != held)),
                "indist_F1": float(f1_mat.loc[held, held]),
            })
            print(f"  [{tag:6s}] F1={np.mean(f1s):.3f}±{np.std(f1s):.3f} "
                  f"ROC-AUC={np.mean(rocs):.3f}±{np.std(rocs):.3f} "
                  f"(best single {max(singles_f1):.3f}, in-dist {f1_mat.loc[held, held]:.3f}) "
                  f"({time.time()-t0:.0f}s)")
    out = pd.DataFrame(rows)
    out.to_csv(combined_csv(), index=False)
    print(f"[+] {combined_csv()}")


run_combined_training()
check_table("12 pooled training", "combined_training.csv", MET_TOL)
report("12 pooled training")

## Bước 13: Khoảng tin cậy của ma trận chuyển giao

Mỗi ô của ma trận là một trung bình trên 5 seed; khoảng tin cậy 95% lấy bằng bootstrap trên các dòng test (500 lần, seed cố định), áp vào trung bình đường chéo,
trung bình ngoài đường chéo, khoảng cách giữa chúng, và từng ô. Hai lượt: CatBoost cho F1, random forest cho ROC-AUC (lượt này ghi thêm từng ô, để biết ô nào
dưới mức ngẫu nhiên 0,5 một cách chắc chắn). Hai bảng này làm tròn bốn chữ số, nên được so với dung sai một đơn vị làm tròn (`ROUND4_TOL`).

In [ ]:
def _fit(model: str, seed: int):
    if model == "RandomForest":
        unweighted = os.environ.get("P2_UNWEIGHTED") == "1"
        return make_model(model, seed, {"class_weight": None} if unweighted else {})
    return make_any_model(model, seed)


def collect_scores(seeds: int = XD_SEEDS, model: str = "RandomForest"):
    data = {n: load_corpus(os.path.join(PROC, f)) for n, f in XD_CORPORA.items()}
    names = list(data)
    held = {}
    for i in names:
        for j in names:
            same = i == j
            per_seed = []
            t0 = time.time()
            for s in range(seeds):
                if same:
                    tr, te = in_dataset_split(data[i], s)
                else:
                    tr, te = data[i], data[j]
                m = _fit(model, s).fit(tr[COMPPHISH], tr["y"])
                sc = m.predict_proba(te[COMPPHISH])[:, 1]
                per_seed.append((te["y"].to_numpy().astype(int), sc))
            held[(i, j)] = per_seed
            f1 = np.mean([f1_score(y, sc >= 0.5, zero_division=0) for y, sc in per_seed])
            roc = np.mean([roc_auc_score(y, sc) for y, sc in per_seed])
            print(f"[i] {i:12s} -> {j:12s} F1={f1:.3f} ROC-AUC={roc:.3f} ({time.time()-t0:.0f}s)",
                  flush=True)
    return names, held


def _cell_metrics(y, sc, idx):
    yb, sb = y[idx], sc[idx]
    if len(np.unique(yb)) < 2:
        return np.nan, np.nan
    return (f1_score(yb, sb >= 0.5, zero_division=0), roc_auc_score(yb, sb))


def bootstrap(names, held, boot: int, rng_seed: int = 0):
    rng = np.random.default_rng(rng_seed)
    keys = list(held)
    out = {k: {"F1": [], "ROC-AUC": []} for k in keys}
    for b in range(boot):
        for k in keys:
            f1s, rocs = [], []
            for y, sc in held[k]:
                idx = rng.integers(0, len(y), len(y))
                f1, roc = _cell_metrics(y, sc, idx)
                f1s.append(f1); rocs.append(roc)
            out[k]["F1"].append(np.nanmean(f1s))
            out[k]["ROC-AUC"].append(np.nanmean(rocs))
        if (b + 1) % 50 == 0:
            print(f"    ... {b+1}/{boot} replicates", flush=True)
    return out


def summarise(names, held, reps, boot):
    diag = [(n, n) for n in names]
    off = [k for k in reps if k[0] != k[1]]
    rows = []
    point = {}
    for metric in ("F1", "ROC-AUC"):
        obs = {}
        for k, per_seed in held.items():
            fn = (lambda y, sc: f1_score(y, sc >= 0.5, zero_division=0)) if metric == "F1" \
                else (lambda y, sc: roc_auc_score(y, sc))
            obs[k] = float(np.mean([fn(y, sc) for y, sc in per_seed]))
        d = float(np.mean([obs[k] for k in diag]))
        o = float(np.mean([obs[k] for k in off]))
        point[metric] = {"diag": d, "off": o, "gap": d - o, "cells": obs}
        dv = np.array([np.mean([reps[k][metric][b] for k in diag]) for b in range(boot)])
        ov = np.array([np.mean([reps[k][metric][b] for k in off]) for b in range(boot)])
        for label, series, pt in (("diagonal", dv, d), ("off-diagonal", ov, o),
                                  ("gap", dv - ov, d - o)):
            lo, hi = np.percentile(series, [2.5, 97.5])
            rows.append({"metric": metric, "quantity": label, "point": round(pt, 4),
                         "ci_lo": round(float(lo), 4), "ci_hi": round(float(hi), 4),
                         "boot": boot})
        # Per-cell interval for every off-diagonal cell, so "N cells are below chance" can be
        # stated as "N cells whose whole interval is below chance" rather than as a bare count.
        for k in off:
            s = np.array(reps[k][metric])
            lo, hi = np.percentile(s, [2.5, 97.5])
            rows.append({"metric": metric, "quantity": f"cell {k[0]}->{k[1]}",
                         "point": round(obs[k], 4), "ci_lo": round(float(lo), 4),
                         "ci_hi": round(float(hi), 4), "boot": boot})
    return pd.DataFrame(rows), point


ARMS = [("CatBoost", "F1"), ("RandomForest", "ROC-AUC")]


def run_xdata_bootstrap(**overrides):
    args = SimpleNamespace(**{**{'boot': 500, 'seeds': 5, 'out': 'data/processed/p2/p2_xdata_bootstrap.csv'}, **overrides})
    # the forest is fitted unweighted; the shared bootstrap code reads the switch from the environment
    os.environ.setdefault("P2_UNWEIGHTED", "1")
    unweighted = os.environ["P2_UNWEIGHTED"] == "1"
    arms = ARMS if unweighted else [a for a in ARMS if a[0] == "RandomForest"]
    frames = []
    for model, metric in arms:
        print(f"\n=== {model} ({metric} matrix) ===", flush=True)
        names, held = collect_scores(args.seeds, model=model)
        reps = bootstrap(names, held, args.boot)
        df, point = summarise(names, held, reps, args.boot)
        df.insert(0, "model", model)
        frames.append(df[df.metric == metric])
        p = point[metric]
        print(f"    {model}/{metric}: diagonal {p['diag']:.3f}, off-diagonal {p['off']:.3f}, "
              f"gap {p['gap']:.3f}")
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(args.out, index=False)
    print(f"[+] {args.out}")


def run_xdata_bootstrap_rf(**overrides):
    args = SimpleNamespace(**{**{'boot': 500, 'seeds': 5, 'out': 'data/processed/p2/p3_xdata_bootstrap.csv'}, **overrides})
    names, held = collect_scores(args.seeds)
    reps = bootstrap(names, held, args.boot)
    df, point = summarise(names, held, reps, args.boot)
    df.to_csv(args.out, index=False)
    print(f"[+] {args.out}")
    for metric in ("F1", "ROC-AUC"):
        p = point[metric]
        print(f"    {metric}: diagonal {p['diag']:.3f}, off-diagonal {p['off']:.3f}, "
              f"gap {p['gap']:.3f}")


ROUND4_TOL = 1.5e-4   # these tables are rounded to 4 decimals: one step in the last digit is 1e-4
run_xdata_bootstrap()
check_table("13 transfer bootstrap", "p2_xdata_bootstrap.csv", ROUND4_TOL)
run_xdata_bootstrap_rf()
check_table("13 transfer bootstrap", "p3_xdata_bootstrap.csv", ROUND4_TOL)
report("13 transfer bootstrap")

## Bước 14: SHAP và trọng số lớp

Đặc trưng nào random forest dựa vào trên PhishVN, và câu trả lời có đổi khi bỏ trọng số lớp không? Với mỗi seed, TreeSHAP tính trên tối đa 2.000 dòng test cân bằng
hai lớp; mỗi đặc trưng có độ lớn đóng góp trung bình, tỷ trọng trong tổng, và hướng (tương quan giữa giá trị đặc trưng và đóng góp của nó).

In [ ]:
def attribution(df, seeds, n_sample, weighting):
    import shap
    params = None if weighting == "balanced" else {"class_weight": None}
    imps, signs = [], []
    for s in range(seeds):
        tr, te = in_dataset_split(df, s)
        m = make_model("RandomForest", s, params).fit(tr[COMPPHISH], tr["y"])
        # the same stratified rows tab_shap explains: the draw depends on the seed only
        sub = te.groupby("y", group_keys=False).apply(
            lambda g: g.sample(min(len(g), n_sample // 2), random_state=s))
        sv = shap.TreeExplainer(m).shap_values(sub[COMPPHISH])
        sv = sv[1] if isinstance(sv, list) else (sv[..., 1] if sv.ndim == 3 else sv)
        imps.append(np.abs(sv).mean(axis=0))
        signs.append([np.corrcoef(sub[c], sv[:, i])[0, 1] if sub[c].nunique() > 1 else np.nan
                      for i, c in enumerate(COMPPHISH)])
    imp = np.mean(imps, axis=0)
    return pd.DataFrame({"feature": COMPPHISH, "weighting": weighting,
                         "mean_abs_shap": imp, "sd_abs_shap": np.std(imps, axis=0),
                         "share": imp / imp.sum(),
                         "direction": np.nanmean(np.array(signs, dtype=float), axis=0)})


SHAP_CORPUS = "data/processed/vn_compphish.csv"


def run_shap_weighting(**overrides):
    a = SimpleNamespace(**{**{'seeds': 5, 'n_sample': 2000, 'out': 'data/processed/p2/p2_shap_weighting.csv'}, **overrides})
    df = load_corpus(SHAP_CORPUS)
    out = pd.concat([attribution(df, a.seeds, a.n_sample, w) for w in ("balanced", "none")])
    out["rank"] = out.groupby("weighting")["share"].rank(ascending=False).astype(int)
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    out.to_csv(a.out, index=False)
    for w, g in out.groupby("weighting"):
        g = g.sort_values("share", ascending=False)
        top = ", ".join(f"{r.feature} {100 * r.share:.1f}%" for r in g.head(5).itertuples())
        print(f"[{w}] top-3 share {100 * g.share.head(3).sum():.1f}%  |  {top}")
    print(f"[+] {a.out}")


run_shap_weighting()
check_table("14 SHAP", "p2_shap_weighting.csv", MET_TOL)
report("14 SHAP")

## Bước 15: Tối ưu siêu tham số

Trên cách chia thời gian của Bước 3 (một mặt nạ benign cố định, seed 0), với CatBoost, HistGB và hồi quy logistic: so mô hình mặc định với mô hình sau khi tìm siêu
tham số. Hai cách tìm có cùng ngân sách số lần thử: Grey Wolf Optimizer (6 con sói, 8 vòng, `gwo_search`) và tìm ngẫu nhiên (`random_search`). Mỗi cấu hình
được chấm bằng PR-AUC kiểm định chéo 3 phần trên tập train; cách nào điểm kiểm định chéo cao hơn thì được dùng (quy tắc chọn đặt trước), rồi mô hình được đo trên
test, trung bình 3 seed.

Bảng kỳ vọng của bài không có cột `search_seed` (chỉ có một lần tìm, seed 0), nên notebook bỏ cột đó trước khi so.

In [ ]:
SPACE = {
    "LogReg": [("C", -3, 3, "logfloat")],
    "RandomForest": [("n_estimators", 100, 600, "int"), ("max_depth", 3, 50, "int"),
                     ("min_samples_leaf", 1, 10, "int"), ("max_features", 0.1, 1.0, "float")],
    "HistGB": [("learning_rate", -2, -0.5, "logfloat"), ("max_leaf_nodes", 15, 255, "int"),
               ("max_depth", 3, 30, "int"), ("l2_regularization", 0.0, 10.0, "float"),
               ("max_iter", 100, 500, "int")],
    "MLP": [("mlpclassifier__alpha", -5, -2, "logfloat"),
            ("mlpclassifier__learning_rate_init", -4, -2, "logfloat")],
    "CatBoost": [("iterations", 100, 600, "int"), ("depth", 4, 10, "int"),
                 ("learning_rate", -2, -0.5, "logfloat"), ("l2_leaf_reg", 1.0, 10.0, "float")],
    "XGBoost": [("n_estimators", 100, 600, "int"), ("max_depth", 3, 10, "int"),
                ("learning_rate", -2, -0.5, "logfloat"), ("subsample", 0.5, 1.0, "float"),
                ("colsample_bytree", 0.5, 1.0, "float")],
}


def _factory(model, seed, params):
    if model in ("CatBoost", "XGBoost", "LightGBM"):
        return make_any_model(model, seed, params)
    return make_model(model, seed, {**(weighting_params(model) or {}), **(params or {})})


def _decode(pos, space):
    params = {}
    for x, (name, lo, hi, kind) in zip(pos, space):
        v = lo + x * (hi - lo)
        if kind == "int":
            params[name] = int(round(v))
        elif kind == "logfloat":
            params[name] = float(10.0 ** v)
        else:
            params[name] = float(v)
    return params


def _fitness(model, params, X, y, seed):
    try:
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        return float(cross_val_score(_factory(model, seed, params), X, y,
                                     scoring="average_precision", cv=cv, n_jobs=-1).mean())
    except Exception as exc:
        # Invalid candidates may occur near a search-space boundary, but silently
        # swallowing every failure makes a broken estimator indistinguishable from
        # a genuinely poor configuration.
        warnings.warn(
            f"fitness evaluation failed for {model} with {params}: {exc}",
            RuntimeWarning,
            stacklevel=2,
        )
        return -1.0


def gwo_budget(n_agents, iters):
    return n_agents * (iters + 1)


def _gwo_step(wolves, leaders, a, rng):
    old = np.asarray(wolves, dtype=float)
    estimates = []
    for lead in np.asarray(leaders, dtype=float):
        r1 = rng.random(old.shape)
        r2 = rng.random(old.shape)
        A = 2.0 * a * r1 - a
        C = 2.0 * r2
        estimates.append(lead - A * np.abs(C * lead - old))
    return np.clip(np.mean(estimates, axis=0), 0.0, 1.0)


def _update_leaders(leader_pos, leader_fit, wolves, fit):
    candidate_pos = np.vstack((leader_pos, wolves))
    candidate_fit = np.concatenate((leader_fit, fit))
    order = np.argsort(-candidate_fit, kind="stable")[:3]
    return candidate_pos[order].copy(), candidate_fit[order].copy()


def random_search(model, X, y, seed=0, budget=48):
    space = SPACE.get(model, [])
    if not space:
        return {}, float("nan")
    rng = np.random.default_rng(seed)
    best_p, best_f = {}, -1.0
    for _ in range(budget):
        p = _decode(rng.random(len(space)), space)
        f = _fitness(model, p, X, y, seed)
        if f > best_f:
            best_p, best_f = p, f
    return best_p, best_f


def gwo_search(model, X, y, seed=0, n_agents=6, iters=8):
    space = SPACE.get(model, [])
    if not space:
        return {}, float("nan")
    if n_agents < 3:
        raise ValueError("GWO requires at least three agents (alpha, beta, delta)")
    if iters < 1:
        raise ValueError("GWO requires at least one iteration")
    d = len(space)
    rng = np.random.default_rng(seed)
    wolves = rng.random((n_agents, d))
    fit = np.array([_fitness(model, _decode(w, space), X, y, seed) for w in wolves])
    # alpha/beta/delta = three best (we maximise, so take the largest)
    order = np.argsort(-fit)
    leaders = wolves[order[:3]].copy()
    leader_fit = fit[order[:3]].copy()
    for t in range(iters):
        a = 2 - 2 * t / max(1, iters - 1)  # linearly 2 -> 0
        wolves = _gwo_step(wolves, leaders, a, rng)
        fit = np.array([_fitness(model, _decode(w, space), X, y, seed) for w in wolves])
        leaders, leader_fit = _update_leaders(leaders, leader_fit, wolves, fit)
    return _decode(leaders[0], space), float(leader_fit[0])


HPO_FAMILIES = ["CatBoost", "HistGB", "LogReg"]
N_AGENTS, ITERS = 6, 8


def run_hpo(**overrides):
    args = SimpleNamespace(**{**{'families': None, 'seeds': 3, 'search_seeds': 1, 'search_seed_start': 0, 'out': 'data/processed/p2/p2_hpo.csv'}, **overrides})
    if args.families is None:
        args.families = HPO_FAMILIES
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    cut = int(len(ph) * 0.70)
    ph_tr, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr.rdom))]
    rng = np.random.RandomState(0)
    bmask = rng.rand(len(be)) < 0.70
    tr = pd.concat([ph_tr, be[bmask]])
    te = pd.concat([ph_te, be[~bmask]])
    Xtr, ytr = tr[feats].to_numpy(float), tr["y"].to_numpy(int)
    Xte, yte = te[feats].to_numpy(float), te["y"].to_numpy(int)
    print(f"phishing-temporal: train={len(tr)} test={len(te)}")
    def eval_test(name, params):
        vals = []
        for s in range(args.seeds):
            m = _factory(name, s, params).fit(Xtr, ytr)
            vals.append(_metrics(yte, m.predict_proba(Xte)[:, 1]))
        return {k: (float(np.mean([v[k] for v in vals])),
                    float(np.std([v[k] for v in vals]))) for k in vals[0]}
    budget = gwo_budget(N_AGENTS, ITERS)
    rows = []
    def checkpoint():
        out_dir = os.path.dirname(args.out)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        pd.DataFrame(rows).to_csv(args.out, index=False)
    for name in args.families:
        res = eval_test(name, {})
        print(f"[default {name}] " + " ".join(f"{k}={m:.3f}±{s:.3f}" for k, (m, s) in res.items()))
        rows.append({"family": name, "config": "default", "cv_prauc": np.nan, "params": "{}",
                     **{k: m for k, (m, s) in res.items()},
                     **{k + "_std": s for k, (m, s) in res.items()}})
        checkpoint()

        for search_seed in range(args.search_seed_start,
                                 args.search_seed_start + args.search_seeds):
            t0 = time.time()
            gp, gf = gwo_search(name, Xtr, ytr, seed=search_seed,
                                n_agents=N_AGENTS, iters=ITERS)
            rp, rf = random_search(name, Xtr, ytr, seed=search_seed, budget=budget)
            adopted = gf >= rf  # pre-specified, equal-budget selection rule
            best_params, src = (gp, "gwo") if adopted else (rp, "random-search")
            res = eval_test(name, best_params)
            print(f"[tuned  {name} seed={search_seed}] CV: gwo={gf:.4f} random={rf:.4f} -> "
                  f"adopted {src} ({time.time()-t0:.0f}s, budget={budget})")
            print("               " + " ".join(f"{k}={m:.3f}±{s:.3f}" for k, (m, s) in res.items()))
            rows.append({"family": name, "config": f"tuned({src})", "search_seed": search_seed,
                         "cv_prauc": max(gf, rf), "cv_prauc_gwo": gf,
                         "cv_prauc_random": rf, "budget": budget,
                         "params": str(best_params),
                         **{k: m for k, (m, s) in res.items()},
                         **{k + "_std": s for k, (m, s) in res.items()}})
            checkpoint()
    print(f"[+] -> {args.out}")


run_hpo()
pd.read_csv(f"{RES}/p2_hpo.csv").drop(columns=["search_seed"]).to_csv(f"{RES}/p2_hpo.csv", index=False)
check_table("15 hyperparameter search", "p2_hpo.csv", MET_TOL)
report("15 hyperparameter search")

## Bước 16: Stacking

Các bài trước về URL lừa đảo hay dùng stacking: vài mô hình nền, rồi một mô hình thứ hai học từ xác suất ngoài mẫu của chúng (`make_stacker`). Bước này chạy công
thức đó trên hai giao thức của Bước 3: random forest + XGBoost, có và không có bước chọn đặc trưng lai (`hybrid_select`, giữ 12 đặc trưng); năm tổ hợp khác
(XGBoost + LightGBM, CatBoost + HistGB, CatBoost + hồi quy logistic, CatBoost + MLP, CatBoost + HistGB + hồi quy logistic + MLP); và CatBoost + hồi quy logistic
với 20 seed để so có cặp với Bước 5.

In [ ]:
def hybrid_select(Xtr: np.ndarray, ytr: np.ndarray, names: list[str], k: int,
                  seed: int) -> list[int]:
    from sklearn.feature_selection import mutual_info_classif
    mi = mutual_info_classif(Xtr, ytr, random_state=seed)
    order = np.argsort(mi)[::-1]
    with np.errstate(invalid="ignore"):  # zero-variance columns yield NaN correlations
        corr = np.nan_to_num(np.corrcoef(Xtr, rowvar=False), nan=0.0)
    kept: list[int] = []
    for i in order:
        if any(abs(corr[i, j]) > 0.90 for j in kept):
            continue
        kept.append(int(i))
        if len(kept) == k:
            break
    return sorted(kept)


ABBREV = {"RandomForest": "RF", "XGBoost": "XGB", "LightGBM": "LGBM", "CatBoost": "CB",
          "HistGB": "HGB", "LogReg": "LR", "MLP": "MLP"}


def make_stacker(bases: list[str], seed: int):
    from sklearn.ensemble import StackingClassifier
    from sklearn.linear_model import LogisticRegression
    return StackingClassifier(
        # members are configured like the single-family rows they are compared with (uniform
        # weighting unless P2_UNWEIGHTED=0), so the stack adds a combiner and nothing else
        estimators=[(ABBREV[b], make_any_model(b, seed, weighting_params(b))) for b in bases],
        final_estimator=LogisticRegression(max_iter=1000),
        stack_method="predict_proba", cv=5, n_jobs=-1)


def run_stacking(**overrides):
    args = SimpleNamespace(**{**{'seeds': 5, 'k': 12, 'bases': ['RandomForest+XGBoost'], 'cut': 0.7, 'test_after': None, 'out': 'data/processed/p2/p2_stacking_baseline.csv', 'curves': 'data/processed/p2/p2_pr_curves_stacking.csv', 'hfs': False}, **overrides})
    combos = [c.split("+") for c in args.bases]
    for combo in combos:
        for b in combo:
            if b not in ABBREV:
                raise SystemExit(f"unknown family in --bases: {b}")
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, n_leaked = split_phishing(ph, args.cut, args.test_after)
    if ph_te.empty:
        raise SystemExit(f"no dated phishing after {args.test_after}: the refresh window has "
                         f"not arrived (corpus max date {ph.date.max().date()})")
    print(f"phishing dated: {len(ph)}  train<= {ph_tr.date.max().date()} ({len(ph_tr)})  "
          f"test>= {ph_te.date.min().date()} ({len(ph_te)}; {n_leaked} guard-dropped)  "
          f"benign pool: {len(be)}  features: {len(feats)}")
    rows, curves = [], []
    for proto in ("temporal_strict", "random_same_rows"):
        for s in range(args.seeds):
            rng = np.random.RandomState(s)
            bmask = rng.rand(len(be)) < args.cut
            if proto == "temporal_strict":
                tr = pd.concat([ph_tr, be[bmask]])
                te = pd.concat([ph_te, be[~bmask]])
            else:
                pool = pd.concat([ph_tr, ph_te])
                pmask = rng.rand(len(pool)) < (len(ph_tr) / (len(ph_tr) + len(ph_te)))
                tr = pd.concat([pool[pmask], be[bmask]])
                te = pd.concat([pool[~pmask], be[~bmask]])
            Xtr, ytr = tr[feats].to_numpy(float), tr["y"].to_numpy(int)
            Xte, yte = te[feats].to_numpy(float), te["y"].to_numpy(int)

            sel = hybrid_select(Xtr, ytr, feats, args.k, s) if args.hfs else []
            for combo in combos:
                tag = "Stack[" + "+".join(ABBREV[b] for b in combo) + "]"
                variants = [(tag, list(range(len(feats))))]
                if args.hfs:
                    variants.append((tag + "+HFS", sel))
                for name, cols in variants:
                    t0 = time.time()
                    m = make_stacker(combo, s)
                    m.fit(Xtr[:, cols], ytr)
                    score = m.predict_proba(Xte[:, cols])[:, 1]
                    met = _metrics(yte, score)
                    met.update({"family": name, "seed": s, "protocol": proto,
                                "fit_seconds": round(time.time() - t0, 2),
                                "n_features": len(cols),
                                "features": "" if len(cols) == len(feats) else
                                            "|".join(feats[i] for i in cols)})
                    rows.append(met)
                    curves.append({"family": name, "protocol": proto, "seed": s,
                                   "precision": pr_curve_row(yte, score)})
                    print(f"  {name:<22} {proto:<17} seed={s} F1={met['F1']:.3f} "
                          f"PR-AUC={met['PR-AUC']:.3f} FPR@R0.90={met['FPR@R0.90']:.3f} "
                          f"({met['fit_seconds']}s, {len(cols)} feats)")
            if args.hfs:
                print(f"    HFS kept: {', '.join(feats[i] for i in sel)}")
    out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    out.to_csv(args.out, index=False)
    write_curves(curves, args.curves)
    print(f"\n[+] {len(out)} runs -> {args.out}")
    print(out.groupby(["family", "protocol"])[["F1", "PR-AUC", "ROC-AUC", "FPR@R0.90"]]
             .mean().round(4).to_string())


run_stacking(hfs=True)
check_table("16 stacking", "p2_stacking_baseline.csv", MET_TOL)
run_stacking(bases=["XGBoost+LightGBM", "CatBoost+HistGB", "CatBoost+LogReg", "CatBoost+MLP", "CatBoost+HistGB+LogReg+MLP"],
             out=f"{RES}/p2_stacking_combos.csv", curves=f"{RES}/p2_pr_curves_stacking_combos.csv")
check_table("16 stacking", "p2_stacking_combos.csv", MET_TOL)
run_stacking(seeds=20, bases=["CatBoost+LogReg"], out=f"{RES}/p2_stacking_cblr_k20.csv", curves="")
check_table("16 stacking", "p2_stacking_cblr_k20.csv", MET_TOL)
report("16 stacking")

## Bước 17: Có dự báo được độ trôi không

Ba phép đo trên cách chia thời gian:
- `e1_decay`: chia phần train theo thời gian thành các cửa sổ, đo hiệu năng của mô hình theo khoảng cách thời gian tới tập test; độ dốc suy giảm của một họ có
  tiên đoán được họ đó tụt bao nhiêu ở Bước 3 không;
- `e2_matrix`: một bộ phân loại "train hay test" giữa từng cặp khối thời gian (W1 tới W4 và TEST); AUC càng cao thì hai khối càng khác nhau;
- `e3_novelty`: độ mới từ vựng của URL test so với train có báo trước được chỗ CatBoost sai không.

Rồi FCTS: stacking trong đó mô hình thứ hai học theo chuỗi thời gian (forward chaining), so với cùng thủ tục khi thứ tự thời gian bị xáo trộn và khi chia fold
ngẫu nhiên.

Bảng suy giảm (`e1_decay`) đo PR-AUC trên từng cửa sổ thời gian nhỏ (vài trăm dòng). Xác suất của random forest là trung bình 300 cây nên có nhiều giá trị bằng
nhau; cộng song song phá các thế hoà đó theo thứ tự khác nhau giữa hai lần chạy, và trên cửa sổ nhỏ PR-AUC nhích tới cỡ 10⁻⁴. Bảng này vì vậy được so với
`THR_TOL` = 10⁻³.

In [ ]:
OUT_DECAY = "data/processed/p2/p2_forecastability_decay.csv"


OUT_MATRIX = "data/processed/p2/p2_forecastability_shiftmatrix.csv"


OUT_NOVELTY = "data/processed/p2/p2_forecastability_novelty.csv"


N_WIN = 4


def split_canonical(df, cut=0.70):
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    k = int(len(ph) * cut)
    ph_tr, ph_te = ph.iloc[:k], ph.iloc[k:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr.rdom))]
    return feats, ph_tr.reset_index(drop=True), ph_te.reset_index(drop=True), be


def guarded_windows(ph_rest, seen):
    wins, seen = [], set(seen)
    for i in range(N_WIN):
        lo, hi = int(len(ph_rest) * i / N_WIN), int(len(ph_rest) * (i + 1) / N_WIN)
        w = ph_rest.iloc[lo:hi]
        wins.append(w[~w.rdom.isin(seen)])
        seen |= set(w.rdom)
    return wins


def e1_decay(feats, ph_tr, be, seeds):
    from sklearn.metrics import average_precision_score, roc_auc_score
    rows = []
    for origin in (0.50, 0.60):
        half = int(len(ph_tr) * origin)
        ph_fit, ph_rest = ph_tr.iloc[:half], ph_tr.iloc[half:].reset_index(drop=True)
        wins = guarded_windows(ph_rest, set(ph_fit.rdom))
        t0 = ph_fit.date.max()
        horiz = [float(w.date.map(lambda d: (d - t0).days).median()) for w in wins]
        print(f"[E1] origin={origin} fit={len(ph_fit)} windows={[len(w) for w in wins]} "
              f"median horizon={[round(h) for h in horiz]}d")
        for s in seeds:
            rng = np.random.RandomState(s)
            bmask = rng.rand(len(be)) < 0.70
            be_tr, be_ev = be[bmask], be[~bmask]
            Xtr = pd.concat([ph_fit, be_tr])[feats].to_numpy(float)
            ytr = np.r_[np.ones(len(ph_fit)), np.zeros(len(be_tr))].astype(int)
            for fam in FAMILIES:
                m = make_any_model(fam, s, weighting_params(fam))
                m.fit(Xtr, ytr)
                sc_be = m.predict_proba(be_ev[feats].to_numpy(float))[:, 1]
                for k, w in enumerate(wins):
                    sc_ph = m.predict_proba(w[feats].to_numpy(float))[:, 1]
                    y = np.r_[np.ones(len(w)), np.zeros(len(be_ev))]
                    sc = np.r_[sc_ph, sc_be]
                    rows.append({"origin": origin, "family": fam, "seed": s, "win": k,
                                 "horizon_d": horiz[k], "n_ph": len(w),
                                 "PR-AUC": average_precision_score(y, sc),
                                 "ROC-AUC": roc_auc_score(y, sc)})
            print(f"[E1] origin={origin} seed={s} done")
    out = pd.DataFrame(rows)
    out.to_csv(OUT_DECAY, index=False)

    # verdict: internal decay ranking vs the canonical gap ranking, per origin x metric
    from scipy.stats import spearmanr
    canon = pd.read_csv("data/processed/p2/p2_temporal_strict.csv")
    g = canon.groupby(["family", "protocol"])["PR-AUC"].mean().unstack()
    gap = (g["random_same_rows"] - g["temporal_strict"])
    for origin in (0.50, 0.60):
        for metric in ("PR-AUC", "ROC-AUC"):
            slopes = {}
            sub = out[out.origin == origin]
            for fam in FAMILIES:
                gg = sub[sub.family == fam].groupby("win")[["horizon_d", metric]].mean()
                slopes[fam] = np.polyfit(gg["horizon_d"], gg[metric], 1)[0]
            s_ = pd.Series(slopes)
            rho, p = spearmanr(-s_[gap.index], gap)  # larger decay should mean larger gap
            print(f"[E1] origin={origin} {metric}: Spearman(internal decay vs canonical gap) "
                  f"rho={rho:+.3f} p={p:.3f}")
    print(f"[+] {len(out)} rows -> {OUT_DECAY}")


def e2_matrix(feats, ph_tr, ph_te):
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import cross_val_predict
    first = ph_tr.drop_duplicates("rdom", keep="first").reset_index(drop=True)
    blocks, names = [], []
    for i in range(N_WIN):
        lo, hi = int(len(first) * i / N_WIN), int(len(first) * (i + 1) / N_WIN)
        blocks.append(first.iloc[lo:hi])
        names.append(f"W{i + 1}")
    blocks.append(ph_te.drop_duplicates("rdom", keep="first"))
    names.append("TEST")
    med = [b.date.median() for b in blocks]
    rows = []
    for i in range(len(blocks)):
        for j in range(i + 1, len(blocks)):
            X = pd.concat([blocks[i], blocks[j]])[feats].to_numpy(float)
            y = np.r_[np.zeros(len(blocks[i])), np.ones(len(blocks[j]))].astype(int)
            oof = cross_val_predict(HistGradientBoostingClassifier(random_state=0), X, y,
                                    cv=5, method="predict_proba")[:, 1]
            auc = roc_auc_score(y, oof)
            dd = abs((med[j] - med[i]).days)
            rows.append({"a": names[i], "b": names[j], "auc": auc, "delta_days": dd,
                         "n_a": len(blocks[i]), "n_b": len(blocks[j])})
            print(f"[E2] {names[i]:>4} vs {names[j]:<4} AUC={auc:.3f} (dt={dd}d, "
                  f"n={len(blocks[i])}/{len(blocks[j])})")
    out = pd.DataFrame(rows)
    out.to_csv(OUT_MATRIX, index=False)
    internal = out[out.a.str.startswith("W") & out.b.str.startswith("W")]
    adj_int = internal[internal.b.str[1].astype(int)
                       - internal.a.str[1].astype(int) == 1]["auc"].mean()
    bound = out[(out.a == "W4") & (out.b == "TEST")]["auc"].iloc[0]
    print(f"[E2] adjacent internal mean AUC={adj_int:.3f}  vs  W4->TEST boundary AUC={bound:.3f}")
    print(f"[+] {len(out)} pairs -> {OUT_MATRIX}")


def e3_novelty(feats, ph_tr, ph_te, be, seeds):
    from sklearn.metrics import roc_auc_score
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import StandardScaler
    rows = []
    for s in seeds:
        rng = np.random.RandomState(s)
        bmask = rng.rand(len(be)) < 0.70
        tr = pd.concat([ph_tr, be[bmask]])
        te = pd.concat([ph_te, be[~bmask]])
        Xtr, ytr = tr[feats].to_numpy(float), tr["y"].to_numpy(int)
        Xte, yte = te[feats].to_numpy(float), te["y"].to_numpy(int)
        sc = StandardScaler().fit(Xtr)
        nn = NearestNeighbors(n_neighbors=10).fit(sc.transform(Xtr))
        u = nn.kneighbors(sc.transform(Xte))[0].mean(axis=1)
        for fam in ("CatBoost", "LogReg"):
            m = make_any_model(fam, s, weighting_params(fam))
            m.fit(Xtr, ytr)
            p = m.predict_proba(Xte)[:, 1]
            wrong = ((p >= 0.5).astype(int) != yte)
            ph_side = yte == 1
            rows.append({"seed": s, "family": fam,
                         "auc_all": roc_auc_score(wrong, u),
                         "auc_phish": roc_auc_score(wrong[ph_side], u[ph_side]),
                         "err_rate": float(wrong.mean())})
            print(f"[E3] seed={s} {fam:<9} AUC(u;wrong) all={rows[-1]['auc_all']:.3f} "
                  f"phish-only={rows[-1]['auc_phish']:.3f}")
    out = pd.DataFrame(rows)
    out.to_csv(OUT_NOVELTY, index=False)
    m = out.groupby("family")[["auc_all", "auc_phish"]].mean().round(3)
    print(f"[E3] means:\n{m.to_string()}")
    print("[E3] pre-set rule: gated backoff stays alive only if CatBoost AUC >= 0.65")
    print(f"[+] {len(out)} rows -> {OUT_NOVELTY}")


def run_forecastability(**overrides):
    args = SimpleNamespace(**{**{'seeds': 5, 'only': None}, **overrides})
    feats, ph_tr, ph_te, be = split_canonical(load())
    print(f"canonical split: ph_tr={len(ph_tr)} ph_te={len(ph_te)} benign={len(be)}")
    seeds = range(args.seeds)
    if args.only in (None, "e2"):
        e2_matrix(feats, ph_tr, ph_te)
    if args.only in (None, "e3"):
        e3_novelty(feats, ph_tr, ph_te, be, seeds)
    if args.only in (None, "e1"):
        e1_decay(feats, ph_tr, be, seeds)


VARIANTS = ("random_fold", "forward_chained", "shuffled_order")


def blocked_folds(order: np.ndarray, n_splits: int = 5):
    from sklearn.model_selection import TimeSeriesSplit
    for tr_pos, oo_pos in TimeSeriesSplit(n_splits=n_splits).split(order):
        yield order[tr_pos], order[oo_pos]


def oof_matrix(bases, X, y, folds):
    P = np.full((len(y), len(bases)), np.nan)
    for tr_idx, oo_idx in folds:
        for j, name in enumerate(bases):
            m = make_any_model(name, 0, weighting_params(name))
            m.fit(X[tr_idx], y[tr_idx])
            P[oo_idx, j] = m.predict_proba(X[oo_idx])[:, 1]
    return P, ~np.isnan(P).any(axis=1)


def chain_order(tr: pd.DataFrame, rng: np.random.RandomState) -> np.ndarray:
    n = len(tr)
    pos = np.empty(n, dtype=float)
    dated = tr["date"].notna().to_numpy()
    if dated.any():
        r = tr.loc[dated, "date"].rank(method="first").to_numpy()
        pos[dated] = (r - 0.5) / dated.sum()
    pos[~dated] = rng.rand((~dated).sum())
    return np.argsort(pos, kind="stable")


def run_fcts(**overrides):
    args = SimpleNamespace(**{**{'seeds': 5, 'bases': 'CatBoost+LogReg', 'cut': 0.7, 'splits': 5, 'out': 'data/processed/p2/p2_fcts.csv'}, **overrides})
    bases = args.bases.split("+")
    for b in bases:
        if b not in ABBREV:
            raise SystemExit(f"unknown family in --bases: {b}")
    tag = "Stack[" + "+".join(ABBREV[b] for b in bases) + "]"
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, n_leaked = split_phishing(ph, args.cut, None)
    print(f"phishing dated: {len(ph)}  train {len(ph_tr)}  test {len(ph_te)} "
          f"({n_leaked} guard-dropped)  benign pool: {len(be)}  features: {len(feats)}")
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    rows = []
    for proto in ("temporal_strict", "random_same_rows"):
        for s in range(args.seeds):
            rng = np.random.RandomState(s)
            bmask = rng.rand(len(be)) < args.cut
            if proto == "temporal_strict":
                tr = pd.concat([ph_tr, be[bmask]])
                te = pd.concat([ph_te, be[~bmask]])
            else:
                pool = pd.concat([ph_tr, ph_te])
                pmask = rng.rand(len(pool)) < (len(ph_tr) / (len(ph_tr) + len(ph_te)))
                tr = pd.concat([pool[pmask], be[bmask]])
                te = pd.concat([pool[~pmask], be[~bmask]])
            tr, te = tr.reset_index(drop=True), te.reset_index(drop=True)
            Xtr, ytr = tr[feats].to_numpy(float), tr["y"].to_numpy(int)
            Xte, yte = te[feats].to_numpy(float), te["y"].to_numpy(int)

            # Prediction-time bases: fitted once on all of train and shared by the variants, so
            # the only thing that varies is the matrix the meta-learner was fitted on.
            Pte = np.column_stack([make_any_model(b, s, weighting_params(b)).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
                                   for b in bases])
            order = chain_order(tr, np.random.RandomState(1000 + s))

            for variant in VARIANTS:
                t0 = time.time()
                if variant == "random_fold":
                    folds = list(StratifiedKFold(args.splits, shuffle=True,
                                                 random_state=s).split(Xtr, ytr))
                elif variant == "forward_chained":
                    folds = list(blocked_folds(order, args.splits))
                else:
                    folds = list(blocked_folds(rng.permutation(len(ytr)), args.splits))
                P, cov = oof_matrix(bases, Xtr, ytr, folds)
                meta = LogisticRegression(max_iter=1000).fit(P[cov], ytr[cov])
                met = _metrics(yte, meta.predict_proba(Pte)[:, 1])
                met.update({"family": tag, "variant": variant, "protocol": proto, "seed": s,
                            "meta_rows": int(cov.sum()),
                            "fit_seconds": round(time.time() - t0, 2)})
                met.update({f"coef_{ABBREV[b]}": round(float(c), 3)
                            for b, c in zip(bases, meta.coef_[0])})
                rows.append(met)
                print(f"  {variant:<16} {proto:<17} seed={s} PR-AUC={met['PR-AUC']:.4f} "
                      f"F1={met['F1']:.3f} meta_n={met['meta_rows']} "
                      f"coef={np.round(meta.coef_[0], 2)} ({met['fit_seconds']}s)")
    out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    out.to_csv(args.out, index=False)
    print(f"\n[+] {len(out)} runs -> {args.out}")
    print(out.groupby(["protocol", "variant"])[["PR-AUC", "F1", "FPR@R0.90"]]
             .mean().round(4).to_string())
    t = out[out.protocol == "temporal_strict"].pivot(index="seed", columns="variant",
                                                     values="PR-AUC")
    vs_shuf = corrected_paired_t(t.forward_chained - t.shuffled_order)
    vs_rand = corrected_paired_t(t.forward_chained - t.random_fold)
    print(f"\ntemporal: forward_chained - shuffled_order  {vs_shuf['mean']:+.4f} "
          f"({vs_shuf['wins']}/{vs_shuf['k']} seeds, p={vs_shuf['p']:.3f})")
    print(f"temporal: forward_chained - random_fold     {vs_rand['mean']:+.4f} "
          f"({vs_rand['wins']}/{vs_rand['k']} seeds, p={vs_rand['p']:.3f})")
    beats = vs_shuf["mean"] > 0 and vs_shuf["p"] <= 0.05 and vs_rand["mean"] > 0
    print("E5: " + ("FORWARD CHAINING HELPS — the order carries signal, follow it up"
                    if beats else
                    "NULL — forward-chained OOF is not better than the shuffled-order control; "
                    "the arrow of time inside the training window buys the meta-learner nothing"))


run_forecastability()
check_table("17 forecastability", "p2_forecastability_decay.csv", THR_TOL)
for f in ("p2_forecastability_shiftmatrix.csv", "p2_forecastability_novelty.csv"):
    check_table("17 forecastability", f, MET_TOL)
run_fcts()
check_table("17 forecastability", "p2_fcts.csv", MET_TOL)
report("17 forecastability")

## Bước 18: Kiểm toán nhãn nhiễu

Theo cách "confident learning": xác suất ngoài mẫu của từng URL (kiểm định chéo 5 phần, 5 seed, `oof_proba`); một URL bị gắn cờ khi mô hình tự tin nó thuộc lớp
kia, với ngưỡng của mỗi lớp là xác suất trung bình mà lớp đó nhận (`flags`). Tỷ lệ gắn cờ được tính riêng cho lừa đảo tầng gold, silver, bronze và cho benign.
Nếu nhãn bronze (một nguồn cộng đồng) nhiễu hơn, tỷ lệ của nó sẽ cao hơn gold.

In [ ]:
CORPUS = "data/processed/vn_compphish.csv"
NOISE_CSV = "data/processed/p2/label_noise_audit.csv"


def oof_proba(X, y, seed: int, folds: int):
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.model_selection import StratifiedKFold

    p = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for tr, te in skf.split(X, y):
        m = CalibratedClassifierCV(
            HistGradientBoostingClassifier(random_state=seed), method="isotonic", cv=3)
        m.fit(X[tr], y[tr])
        p[te] = m.predict_proba(X[te])[:, 1]
    return p


def flags(y, p_phish):
    p_benign = 1 - p_phish
    t_phish = p_phish[y == 1].mean()
    t_benign = p_benign[y == 0].mean()
    flagged = np.where(y == 1, p_benign >= t_benign, p_phish >= t_phish)
    return flagged, t_phish, t_benign


def run_label_noise(**overrides):
    args = SimpleNamespace(**{**{'seeds': 5, 'folds': 5}, **overrides})
    df = add_label(pd.read_csv(CORPUS, low_memory=False))
    for c in COMPPHISH:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    X = df[COMPPHISH].to_numpy(float)
    y = df["y"].to_numpy(int)
    tier = df["tier"].fillna("?").to_numpy()
    print(f"[i] corpus: {len(df)} rows; phishing tiers: "
          + ", ".join(f"{t}={int(((tier == t) & (y == 1)).sum())}"
                      for t in ("gold", "silver", "bronze")))
    # groups audited: phishing per tier, plus all benign as the opposite-direction reference
    groups = {t: (tier == t) & (y == 1) for t in ("gold", "silver", "bronze")}
    groups["benign (all)"] = y == 0
    rates = {g: [] for g in groups}
    per_seed_rows = []
    flag_count = np.zeros(len(df))
    for s in range(args.seeds):
        p = oof_proba(X, y, s, args.folds)
        fl, t_p, t_b = flags(y, p)
        flag_count += fl
        for g, mask in groups.items():
            r = float(fl[mask].mean())
            rates[g].append(r)
            per_seed_rows.append({"group": g, "seed": s, "n": int(mask.sum()),
                                  "flag_rate": round(r, 4)})
        print(f"[i] seed {s}: thresholds t_phish={t_p:.3f} t_benign={t_b:.3f}; "
              + " ".join(f"{g}={np.mean(fl[m]):.3f}" for g, m in groups.items()))
    os.makedirs(os.path.dirname(NOISE_CSV), exist_ok=True)
    pd.DataFrame(per_seed_rows).to_csv(NOISE_CSV, index=False)
    print(f"[+] {NOISE_CSV}")
    # a majority-of-seeds flag is the stable per-row verdict; used for the qualitative sample
    stable = flag_count >= (args.seeds / 2)
    ex = df.loc[stable & groups["bronze"], "dom" if "dom" in df else "url"].astype(str)
    print("[i] example stable-flagged bronze domains:", ", ".join(ex.head(12)))
    for g, r in rates.items():
        print(f"  {g:14s} flagged {100 * np.mean(r):.1f}% +- {100 * np.std(r):.1f} (n={int(groups[g].sum()):,})")


run_label_noise()
check_table("18 label noise", "label_noise_audit.csv", MET_TOL)
report("18 label noise")

## Bước 19: Độ nhạy theo tầng nhãn

Hai giao thức của Bước 3, CatBoost và hồi quy logistic, 5 seed: F1 chung, và tỷ lệ bắt được URL lừa đảo trong từng tầng nhãn. Tầng nhãn gắn liền với nguồn và
thời gian, nên đây là chẩn đoán, không phải tác động nhân quả của tầng.

In [ ]:
SEEDS = 5


def _save(df, relative):
    path = relative
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False, float_format="%.6f")
    print(f"[+] {path}")


def tier_sensitivity(df):
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, _ = split_phishing(ph, 0.70)
    rows = []
    for family in ("CatBoost", "LogReg"):
        for seed in range(SEEDS):
            rng = np.random.RandomState(seed)
            bmask = rng.rand(len(be)) < 0.70
            pool = pd.concat([ph_tr, ph_te]).reset_index(drop=True)
            pmask = rng.rand(len(pool)) < len(ph_tr) / len(pool)
            protocols = {
                "phishing-temporal": (pd.concat([ph_tr, be[bmask]]),
                                       pd.concat([ph_te, be[~bmask]])),
                "random-same-rows": (pd.concat([pool[pmask], be[bmask]]),
                                     pd.concat([pool[~pmask], be[~bmask]])),
            }
            for protocol, (tr, te) in protocols.items():
                model = make_any_model(family, seed, weighting_params(family))
                model.fit(tr[feats].to_numpy(float), tr.y.to_numpy(int))
                score = model.predict_proba(te[feats].to_numpy(float))[:, 1]
                pred = score >= 0.5
                rows.append({"family": family, "protocol": protocol, "seed": seed,
                             "tier": "overall", "n": len(te),
                             "metric": "F1", "value": f1_score(te.y, pred)})
                for tier in ("bronze", "silver", "gold"):
                    mask = (te.y.to_numpy(int) == 1) & te.tier.astype(str).eq(tier).to_numpy()
                    rows.append({"family": family, "protocol": protocol, "seed": seed,
                                 "tier": tier, "n": int(mask.sum()), "metric": "recall",
                                 "value": float(pred[mask].mean()) if mask.any() else np.nan})
    out = pd.DataFrame(rows)
    _save(out, "data/processed/p2/p2_tier_sensitivity.csv")


tier_sensitivity(df_all)
check_table("19 label tiers", "p2_tier_sensitivity.csv", MET_TOL)
report("19 label tiers")

## Bước 20: CNN trên ký tự

Một mạng CNN đọc thẳng chuỗi URL thay cho 21 đặc trưng: mỗi ký tự thành một vector 32 chiều, bốn bộ lọc tích chập độ rộng 3 tới 6 (128 bộ mỗi độ rộng), lấy cực
đại theo chiều dài, rồi một lớp tuyến tính (`make_net`). Bảng chữ cái chỉ lấy từ tập train; 10% tập train làm validation để dừng sớm (tối đa 12 epoch, dừng khi
hai epoch liên tiếp không tốt hơn). Seed cố định cả khởi tạo lẫn thứ tự lô, và mạng chạy trên CPU ở chế độ tất định.

Ba lượt trên hai giao thức của Bước 3: nguyên URL; bỏ phần scheme (`http://`), vì tỷ lệ https khác nhau giữa hai lớp là dấu vết của cách thu thập; và bỏ scheme với
benign chia theo tên miền. Rồi phép thử nguồn benign (`with_source`): huấn luyện với benign chỉ từ một họ nguồn (Tranco, hoặc danh sách Tín Nhiệm Mạng), kiểm tra
với benign từ họ kia, cho CNN, CatBoost và hồi quy logistic, 3 seed và 8 epoch.

Lượt nguyên URL chạy trên CPU như bài, nên được so theo dung sai thông thường. Hai lượt bỏ scheme được bài huấn luyện trên GPU Apple (MPS); trên CPU, phép toán số
thực khác thứ tự nên trọng số học được lệch dần và chỉ số lệch ở chữ số thứ ba, vì vậy được so với dung sai `CNN_TOL` và notebook in mức lệch thật.

In [ ]:
MAX_LEN = 128          # p99 is 43; 20 rows of 53,116 are longer than this and are truncated


EMB = 32


WIDTHS = (3, 4, 5, 6)


FILTERS = 128


DROPOUT = 0.3


BATCH = 256


LR = 1e-3


VAL_FRAC = 0.10        # carved from TRAIN, never from the test window


def encode(urls, vocab):
    out = np.zeros((len(urls), MAX_LEN), dtype=np.int64)
    for i, u in enumerate(urls):
        for j, ch in enumerate(str(u)[:MAX_LEN]):
            out[i, j] = vocab.get(ch, 1)
    return out


def build_vocab(urls):
    chars = sorted({ch for u in urls for ch in str(u)[:MAX_LEN]})
    return {ch: i + 2 for i, ch in enumerate(chars)}


def make_net(vocab_size, seed):
    import torch, torch.nn as nn
    torch.manual_seed(seed)

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb = nn.Embedding(vocab_size + 2, EMB, padding_idx=0)
            self.convs = nn.ModuleList(
                [nn.Conv1d(EMB, FILTERS, w, padding=w // 2) for w in WIDTHS])
            self.drop = nn.Dropout(DROPOUT)
            self.fc = nn.Linear(FILTERS * len(WIDTHS), 1)

        def forward(self, x):
            h = self.emb(x).transpose(1, 2)
            h = torch.cat([c(h).relu().amax(dim=2) for c in self.convs], dim=1)
            return self.fc(self.drop(h)).squeeze(1)

    return Net()


def fit_predict(tr_urls, ytr, te_urls, seed, epochs, patience=2, device="cpu"):
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

    rng = np.random.RandomState(seed)
    vmask = rng.rand(len(tr_urls)) < VAL_FRAC
    vocab = build_vocab(np.asarray(tr_urls)[~vmask])          # TRAIN-only alphabet

    Xtr = torch.from_numpy(encode(np.asarray(tr_urls)[~vmask], vocab))
    Xva = torch.from_numpy(encode(np.asarray(tr_urls)[vmask], vocab))
    Xte = torch.from_numpy(encode(np.asarray(te_urls), vocab))
    ytr_t = torch.from_numpy(np.asarray(ytr)[~vmask].astype(np.float32))
    yva_t = torch.from_numpy(np.asarray(ytr)[vmask].astype(np.float32))

    net = make_net(len(vocab), seed).to(device)
    Xva, yva_t = Xva.to(device), yva_t.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=LR)
    lossf = torch.nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(Xtr, ytr_t), batch_size=BATCH, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))

    best, best_state, bad = float("inf"), None, 0
    for _ in range(epochs):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            lossf(net(xb), yb).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            vloss = lossf(net(Xva), yva_t).item()
        if vloss < best - 1e-4:
            best, bad = vloss, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        # score in slices: the test corpus can be 53k rows and the whole tensor need not sit on
        # the accelerator at once
        out = [torch.sigmoid(net(Xte[i:i + 4096].to(device))).cpu()
               for i in range(0, len(Xte), 4096)]
        return torch.cat(out).numpy()


def run_charcnn(**overrides):
    a = SimpleNamespace(**{**{'seeds': 5, 'epochs': 12, 'cut': 0.7, 'test_after': None, 'out': 'data/processed/p2/p2_charcnn.csv', 'curves': 'data/processed/p2/p2_pr_curves_charcnn.csv', 'device': 'cpu', 'benign_grouped': False, 'strip_scheme': False}, **overrides})
    df = load()
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, n_leaked = split_phishing(ph, a.cut, a.test_after)
    print(f"phishing dated: {len(ph)}  train ({len(ph_tr)})  test ({len(ph_te)}; "
          f"{n_leaked} guard-dropped)  benign pool: {len(be)}")
    rows, curves = [], []
    for proto in ("temporal_strict", "random_same_rows"):
        for s in range(a.seeds):
            rng = np.random.RandomState(s)                 # the canonical per-seed benign mask
            bmask = benign_mask(be, rng, a.cut, a.benign_grouped)
            if proto == "temporal_strict":
                tr = pd.concat([ph_tr, be[bmask]])
                te = pd.concat([ph_te, be[~bmask]])
            else:
                pool = pd.concat([ph_tr, ph_te])
                pmask = rng.rand(len(pool)) < (len(ph_tr) / (len(ph_tr) + len(ph_te)))
                tr = pd.concat([pool[pmask], be[bmask]])
                te = pd.concat([pool[~pmask], be[~bmask]])

            t0 = time.time()
            enc = (lambda xs: [re.sub(r"^[A-Za-z][A-Za-z0-9+.-]*://", "", str(u)) for u in xs]) \
                if a.strip_scheme else (lambda xs: xs)
            score = fit_predict(enc(tr["url"].tolist()), tr["y"].to_numpy(int),
                                enc(te["url"].tolist()), s, a.epochs, device=a.device)
            yte = te["y"].to_numpy(int)
            met = _metrics(yte, score)
            met.update({"family": "CharCNN", "seed": s, "protocol": proto,
                        "fit_seconds": round(time.time() - t0, 2)})
            rows.append(met)
            curves.append({"family": "CharCNN", "protocol": proto, "seed": s,
                           "precision": pr_curve_row(yte, score)})
            print(f"  CharCNN  {proto:<17} seed={s} F1={met['F1']:.3f} "
                  f"PR-AUC={met['PR-AUC']:.3f} FPR@R0.90={met['FPR@R0.90']:.3f} "
                  f"({met['fit_seconds']}s)")
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    pd.DataFrame(rows).to_csv(a.out, index=False)
    if a.curves:
        write_curves(curves, a.curves)
    print(f"[+] {a.out}")


TRANCO = {"tranco", "tranco_vn"}


TINNHIEM = {"tinnhiem_org", "tinnhiem_web"}


def with_source(df):
    src = pd.read_csv(URL_CSV, usecols=["url_norm", "source"], dtype=str)
    return df.merge(src.rename(columns={"url_norm": "url"}), on="url", how="left")


def run_source_probe(**overrides):
    a = SimpleNamespace(**{**{'seeds': 3, 'epochs': 8, 'cut': 0.7, 'families': ['CatBoost', 'LogReg'], 'out': 'data/processed/p2/p2_source_probe.csv'}, **overrides})
    df = with_source(load())
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, _ = split_phishing(ph, a.cut)
    print(f"benign by family: tranco {be.source.isin(TRANCO).sum()}  "
          f"tinnhiem {be.source.isin(TINNHIEM).sum()}  (unmapped {be.source.isna().sum()})")
    rows = []
    for train_fam, name in ((TRANCO, "tranco"), (TINNHIEM, "tinnhiem")):
        be_tr = be[be.source.isin(train_fam)]
        be_te = be[~be.source.isin(train_fam) & be.source.notna()]
        for s in range(a.seeds):
            tr = pd.concat([ph_tr, be_tr])
            te = pd.concat([ph_te, be_te])
            yte = te["y"].to_numpy(int)

            t0 = time.time()
            score = fit_predict(tr["url"].tolist(), tr["y"].to_numpy(int),
                                te["url"].tolist(), s, a.epochs)
            met = _metrics(yte, score)
            met.update({"family": "CharCNN", "seed": s, "train_benign": name,
                        "fit_seconds": round(time.time() - t0, 1)})
            rows.append(met)
            print(f"  CharCNN   train_benign={name:<9} seed={s} F1={met['F1']:.3f} "
                  f"PR-AUC={met['PR-AUC']:.3f} FPR@R0.90={met['FPR@R0.90']:.3f}")

            Xtr, ytr = tr[feats].to_numpy(float), tr["y"].to_numpy(int)
            Xte = te[feats].to_numpy(float)
            for fam in a.families:
                t0 = time.time()
                m = make_any_model(fam, s, weighting_params(fam))
                m.fit(Xtr, ytr)
                sc = m.predict_proba(Xte)[:, 1]
                met = _metrics(yte, sc)
                met.update({"family": fam, "seed": s, "train_benign": name,
                            "fit_seconds": round(time.time() - t0, 1)})
                rows.append(met)
                print(f"  {fam:<9} train_benign={name:<9} seed={s} F1={met['F1']:.3f} "
                      f"PR-AUC={met['PR-AUC']:.3f} FPR@R0.90={met['FPR@R0.90']:.3f}")
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    pd.DataFrame(rows).to_csv(a.out, index=False)
    print(f"[+] {a.out}")


CNN_TOL = 0.03     # networks the paper trained on Apple's GPU (MPS), refitted here on the CPU

run_charcnn()
check_table("20 character CNN", "p2_charcnn.csv", MET_TOL)
run_charcnn(strip_scheme=True, out=f"{RES}/p2_charcnn_noscheme.csv", curves="")
check_table("20 character CNN", "p2_charcnn_noscheme.csv", CNN_TOL)
run_charcnn(strip_scheme=True, benign_grouped=True, out=f"{RES}/p2_charcnn_noscheme_bgroup.csv", curves="")
check_table("20 character CNN", "p2_charcnn_noscheme_bgroup.csv", CNN_TOL)
run_source_probe()
check_table("20 character CNN", "p2_source_probe.csv", MET_TOL)
report("20 character CNN")

## Bước 21: CNN trên ký tự trong ma trận chuyển giao

Ma trận của Bước 11 với mạng CNN thay cho các họ trên đặc trưng: ô đường chéo dùng đúng cách chia trong corpus của Bước 11; ô ngoài đường chéo huấn luyện một mạng
trên toàn bộ corpus nguồn rồi chấm cả ba corpus đích. Bài huấn luyện 40 mạng này trên MPS, nên hai ma trận F1 và ROC-AUC được so với dung sai `CNN_TOL`.

In [ ]:
XD_CNN_CORPORA = [("PhishVN", "data/processed/vn_compphish.csv"),
                  ("PhiUSIIL", "data/processed/external/phiusiil_compphish.csv"),
                  ("ISCXURL2016", "data/processed/external/iscx_compphish.csv"),
                  ("PhishStorm", "data/processed/external/phishstorm_compphish.csv")]


_SCHEME = re.compile(r"^[a-z][a-z0-9+.-]*://", re.I)


def strip_scheme(u) -> str:
    return _SCHEME.sub("", str(u))


def run_charcnn_xdata(**overrides):
    a = SimpleNamespace(**{**{'seeds': 5, 'epochs': 12, 'device': 'cpu', 'out': 'data/processed/p2/p2_charcnn_xdata.csv', 'matrix_prefix': 'data/processed/p2/cross_dataset_'}, **overrides})
    corpora = {}
    for name, path in XD_CNN_CORPORA:
        df = load_corpus(path)
        df["url_s"] = df["url"].map(strip_scheme)
        corpora[name] = df
        print(f"[i] {name}: {len(df)} rows (phishing={int(df.y.sum())}), "
              f"{df.url_s.str.len().median():.0f} chars median")
    names = [n for n, _ in XD_CNN_CORPORA]
    rows = []
    for src in names:
        S = corpora[src]
        for s in range(a.seeds):
            # diagonal: the grouped in-corpus split, same draw as the tabular matrix
            tr, te = in_dataset_split(S, s)
            t0 = time.time()
            sc = fit_predict(tr.url_s.tolist(), tr.y.to_numpy(int), te.url_s.tolist(),
                             s, a.epochs, device=a.device)
            m = _metrics(te.y.to_numpy(int), sc)
            m.update(train=src, test=src, seed=s, n_train=len(tr), n_test=len(te),
                     fit_seconds=round(time.time() - t0, 1), device=a.device)
            rows.append(m)
            print(f"  {src:<12}-> {src:<12} seed={s} F1={m['F1']:.3f} ROC={m['ROC-AUC']:.3f} "
                  f"({m['fit_seconds']}s, in-dist)", flush=True)
            # off-diagonal: one fit on the whole source, scored on each whole target. The
            # network is trained once and only scored three times, so the fit time is charged
            # to the first target and zero to the others; the CSV says so.
            targets = [t for t in names if t != src]
            t0 = time.time()
            net_scores = fit_predict(S.url_s.tolist(), S.y.to_numpy(int),
                                     pd.concat([corpora[t].url_s for t in targets]).tolist(),
                                     s, a.epochs, device=a.device)
            fit_s = round(time.time() - t0, 1)
            off = 0
            for t in targets:
                T = corpora[t]
                sc = net_scores[off:off + len(T)]
                off += len(T)
                m = _metrics(T.y.to_numpy(int), sc)
                m.update(train=src, test=t, seed=s, n_train=len(S), n_test=len(T),
                         fit_seconds=fit_s if t == targets[0] else 0.0, device=a.device)
                rows.append(m)
                print(f"  {src:<12}-> {t:<12} seed={s} F1={m['F1']:.3f} "
                      f"ROC={m['ROC-AUC']:.3f}", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)  # checkpoint after every seed
    out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    out.to_csv(a.out, index=False)
    print(f"[+] {len(out)} rows -> {a.out}")
    if a.matrix_prefix:
        for metric in ("F1", "ROC-AUC"):
            mat = (out.pivot_table(index="train", columns="test", values=metric, aggfunc="mean")
                      .reindex(index=names, columns=names).round(3))
            mat.index.name = None
            path = f"{a.matrix_prefix}{metric}_CharCNN.csv"
            mat.to_csv(path)
            v = mat.to_numpy(float)
            d, o = np.diag(v).mean(), v[~np.eye(len(v), dtype=bool)].mean()
            print(f"[+] {path}  diag={d:.3f} off={o:.3f} gap={d - o:.3f}")
            print(mat.to_string())


run_charcnn_xdata()
check_matrix("21 CNN transfer", "cross_dataset_F1_CharCNN.csv", CNN_TOL)
check_matrix("21 CNN transfer", "cross_dataset_ROC-AUC_CharCNN.csv", CNN_TOL)
report("21 CNN transfer")

## Bước 22: Bỏ dòng trùng và chạy lại

Bộ PhishVN đã công bố ghi 16.855 host hai lần: gần như mọi dòng lừa đảo không có ngày từ nguồn cộng đồng là bản có `www.` của một dòng có ngày cùng host. Bước này
giữ mỗi host một dòng (`dedup`: host đã bỏ scheme và `www.`; ưu tiên dòng có ngày, rồi dòng không có `www.`), còn 36.261 dòng, rồi chạy lại trên corpus đó những
thí nghiệm mà dòng trùng chạm tới: Bước 2, 3, đối chứng có guard, ba ma trận chính, rò rỉ trùng lặp, SHAP, rồi các ma trận can thiệp, huấn luyện gộp và CNN chuyển
giao.

Cách làm: dựng một cây thư mục thứ hai `url_work_dedup/` giống hệt cây chính nhưng `dataset_url.csv` và `vn_compphish.csv` là bản đã bỏ trùng, chuyển vào đó và gọi
lại đúng các hàm của các bước trước, với BLAS một luồng như bài. Kết quả được chép sang `data/processed/p2/dedup_audit/` của cây chính và so với bản kỳ vọng cùng
tên.

In [ ]:
COPIED = ["p2_benchmark", "p2_temporal_strict", "p2_temporal_strict_guarded", "cross_dataset_F1",
          "cross_dataset_ROC-AUC", "cross_dataset_F1_CatBoost", "p2_dup_leakage",
          "p2_shap_weighting"]


TRANSFER = ["cross_dataset_" + m + t for m in ("F1", "ROC-AUC")
            for t in ("_LogReg", "_HistGB", "_pruned", "_LogReg_pruned", "_HistGB_pruned",
                      "_coral", "_LogReg_coral")] + [
    "cross_dataset_F1_CharCNN", "cross_dataset_ROC-AUC_CharCNN", "p2_charcnn_xdata",
    "combined_training"]


def bare_host(urls: pd.Series) -> pd.Series:
    return (urls.astype(str).str.lower().str.replace(r"^[a-z][a-z0-9+.-]*://", "", regex=True)
            .str.replace(r"^www\.", "", regex=True))


def dedup() -> tuple[pd.DataFrame, pd.DataFrame]:
    u = pd.read_csv(os.path.join(PROC, "dataset_url.csv"), low_memory=False)
    c = pd.read_csv(os.path.join(PROC, "vn_compphish.csv"), low_memory=False)
    if len(u) != len(c):
        raise SystemExit("dataset_url.csv and vn_compphish.csv are not row-aligned")
    s = u["collected_at"].astype(str).str.strip().str.slice(0, 10)
    dated = (pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
             .fillna(pd.to_datetime(s, format="%Y-%m-%d", errors="coerce"))).notna()
    host = u["url_norm"].astype(str).str.lower().str.replace(r"^[a-z][a-z0-9+.-]*://", "", regex=True)
    k = pd.DataFrame({"i": np.arange(len(u)), "bare": bare_host(u["url_norm"]),
                      "dated": dated, "www": host.str.startswith("www.")})
    k = k.sort_values(["bare", "dated", "www", "i"], ascending=[True, False, True, True])
    keep = np.sort(k.drop_duplicates("bare")["i"].to_numpy())
    return u.iloc[keep].reset_index(drop=True), c.iloc[keep].reset_index(drop=True)


ud, cd = dedup()
DEDUP_OUT = Path(PROC) / "p2" / "dedup_audit"
DEDUP_OUT.mkdir(parents=True, exist_ok=True)
ud.to_csv(DEDUP_OUT / "dataset_url_dedup.csv", index=False)
cd.to_csv(DEDUP_OUT / "vn_compphish_dedup.csv", index=False)
print(f"kept {len(ud):,} of the corpus's rows, one per host")

TREE = HOME / "url_work_dedup"
(TREE / "data" / "processed" / "p2").mkdir(parents=True, exist_ok=True)
for name in ("external", "first_seen_validation_summary.json"):
    dst = TREE / "data" / "processed" / name
    if not dst.exists():
        os.symlink(Path(PROC) / name, dst)
ud.to_csv(TREE / "data" / "processed" / "dataset_url.csv", index=False)
cd.to_csv(TREE / "data" / "processed" / "vn_compphish.csv", index=False)

os.chdir(TREE)
ROOT = str(TREE)
try:
    with threadpool_limits(1, "blas"):
        run_benchmark()
        run_strict()
        run_strict(guard_control=True, out=f"{RES}/p2_temporal_strict_guarded.csv", curves="")
        run_cross_dataset(metric="F1", out=f"{RES}/cross_dataset_F1.csv")
        run_cross_dataset(metric="ROC-AUC", out=f"{RES}/cross_dataset_ROC-AUC.csv")
        run_cross_dataset(metric="F1", model="CatBoost", out=f"{RES}/cross_dataset_F1_CatBoost.csv")
        run_dup_leakage()
        run_shap_weighting()
        for metric in ("F1", "ROC-AUC"):
            for model, suf in (("LogReg", "_LogReg"), ("HistGB", "_HistGB")):
                run_cross_dataset(metric=metric, model=model, out=f"{RES}/cross_dataset_{metric}{suf}.csv")
            for model, suf in (("RandomForest", ""), ("LogReg", "_LogReg"), ("HistGB", "_HistGB")):
                run_cross_dataset(metric=metric, model=model, drop=PRUNE, out=f"{RES}/cross_dataset_{metric}{suf}_pruned.csv")
            for model, suf in (("RandomForest", ""), ("LogReg", "_LogReg")):
                run_cross_dataset(metric=metric, model=model, adapt="coral", out=f"{RES}/cross_dataset_{metric}{suf}_coral.csv")
        PROC_SAVED, PROC = PROC, os.path.join(ROOT, "data", "processed")
        try:
            run_combined_training()
        finally:
            PROC = PROC_SAVED
        run_charcnn_xdata()
finally:
    os.chdir(WORK)
    ROOT = str(WORK)
for f in COPIED + TRANSFER:
    shutil.copyfile(TREE / "data" / "processed" / "p2" / f"{f}.csv", DEDUP_OUT / f"{f}.csv")

for f in COPIED + TRANSFER:
    name = f"dedup_audit/{f}.csv"
    if f.startswith("cross_dataset_"):
        check_matrix("22 deduplicated corpus", name, CNN_TOL if f.endswith("_CharCNN") else 0.0, path=DEDUP_OUT / f"{f}.csv")
    elif f != "p2_charcnn_xdata":
        check_table("22 deduplicated corpus", name, MET_TOL, path=DEDUP_OUT / f"{f}.csv")
for f in ("dataset_url_dedup.csv", "vn_compphish_dedup.csv"):
    import filecmp
    check("22 deduplicated corpus", f"dedup_audit/{f} byte-identical", filecmp.cmp(DEDUP_OUT / f, EXPECTED / "dedup_audit" / f, shallow=False))
report("22 deduplicated corpus")

## Bước 23: Bỏ tiền tố `www.`

Tiền tố `www.` mang nghĩa ngược nhau ở các corpus: ở PhishVN và ISCX-URL2016 nó gần như chỉ có ở dòng lừa đảo, ở PhiUSIIL nó có ở mọi dòng benign, ở PhishStorm ở
hai phần ba dòng benign. `subdom_cnt`, `dot_cnt` và các đặc trưng độ dài đều đếm nó, nên một mô hình có thể học "có `www.` là lừa đảo" trên corpus này rồi bị chấm
trên corpus mà nó có nghĩa ngược lại. Bước này bỏ tiền tố khỏi mọi host của cả bốn corpus (PhishVN dùng bản mỗi host một dòng của Bước 22), tính lại đặc trưng
(`extract`), ghi tỷ lệ dòng có tiền tố theo corpus và lớp, rồi chạy lại các ma trận chính.

In [ ]:
from collections import Counter
from math import log2
from urllib.parse import urlparse


_EXT = tldextract.TLDExtract(suffix_list_urls=())  # the snapshot bundled with the library, no network


COLUMNS = ["url", "url_len", "dom", "dom_len", "is_ip", "tld", "tld_len", "subdom_cnt",
           "letter_cnt", "digit_cnt", "special_cnt", "eq_cnt", "qm_cnt", "amp_cnt",
           "dot_cnt", "dash_cnt", "under_cnt", "letter_ratio", "digit_ratio", "spec_ratio",
           "is_https", "slash_cnt", "entropy", "path_len", "query_len"]


_IP_RE = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")


def _entropy(s: str) -> float:
    if not s:
        return 0.0
    n = len(s)
    return -sum((c / n) * log2(c / n) for c in Counter(s).values())


def extract(url: str) -> dict:
    url = str(url).strip()
    p = urlparse(url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url)
    host = (p.hostname or "").lower()
    # scheme-stripped URL used for ALL lexical features (scheme-independent)
    core = re.sub(r"^[a-z]+://", "", url, flags=re.I)

    if _EXT is not None:
        ext = _EXT(url if re.match(r"^[a-z]+://", url, re.I) else "http://" + url)
        suffix = ext.suffix
        dom = ext.registered_domain or host
        subdom = ext.subdomain
    else:  # fallback (less accurate for multi-part TLDs)
        parts = host.split(".")
        suffix = parts[-1] if parts else ""
        dom = ".".join(parts[-2:]) if len(parts) >= 2 else host
        subdom = ".".join(parts[:-2]) if len(parts) > 2 else ""

    url_len = len(core)
    letter_cnt = sum(c.isalpha() for c in core)
    digit_cnt = sum(c.isdigit() for c in core)
    special_cnt = url_len - letter_cnt - digit_cnt
    is_ip = 1 if _IP_RE.match(host) else 0
    path_len = len(p.path.strip("/"))
    return {
        "url": url,
        "url_len": url_len,
        "dom": dom,
        "dom_len": len(dom),
        "is_ip": is_ip,
        "tld": suffix,
        "tld_len": len(suffix),
        "subdom_cnt": len([s for s in subdom.split(".") if s]) if subdom else 0,
        "letter_cnt": letter_cnt,
        "digit_cnt": digit_cnt,
        "special_cnt": special_cnt,
        "eq_cnt": core.count("="),
        "qm_cnt": core.count("?"),
        "amp_cnt": core.count("&"),
        "dot_cnt": core.count("."),
        "dash_cnt": core.count("-"),
        "under_cnt": core.count("_"),
        "letter_ratio": letter_cnt / url_len if url_len else 0.0,
        "digit_ratio": digit_cnt / url_len if url_len else 0.0,
        "spec_ratio": special_cnt / url_len if url_len else 0.0,
        "is_https": 1 if url.lower().startswith("https") else 0,
        "slash_cnt": core.count("/"),
        "entropy": _entropy(core),
        "path_len": path_len,
        "query_len": len(p.query),
    }


NOWWW_OUT = os.path.join(PROC, "p2", "nowww")
SOURCES = {"phishvn": os.path.join(PROC, "p2", "dedup_audit", "vn_compphish_dedup.csv"),
           "phiusiil": os.path.join(PROC, "external", "phiusiil_compphish.csv"),
           "iscx": os.path.join(PROC, "external", "iscx_compphish.csv"),
           "phishstorm": os.path.join(PROC, "external", "phishstorm_compphish.csv")}


NAMES = {"phishvn": "PhishVN", "phiusiil": "PhiUSIIL", "iscx": "ISCXURL2016",
         "phishstorm": "PhishStorm"}


WWW = re.compile(r"^((?:[A-Za-z][A-Za-z0-9+.-]*://)?)www\.", re.I)


def is_phish(v) -> bool:
    return str(v).strip().lower() in {"1", "phishing", "phish", "malicious", "bad"}


def build():
    os.makedirs(NOWWW_OUT, exist_ok=True)
    share = []
    for key, path in SOURCES.items():
        df = pd.read_csv(path, low_memory=False)
        url = df["url"].astype(str)
        has = url.map(lambda u: bool(WWW.match(u)))
        y = df["label"].map(is_phish)
        for cls, m in (("phishing", y), ("benign", ~y)):
            share.append({"corpus": NAMES[key], "class": cls, "rows": int(m.sum()),
                          "www_share": float(has[m].mean())})
        feats = pd.DataFrame([extract(WWW.sub(r"\1", u)) for u in url])
        for col in df.columns:
            if col not in feats.columns:
                feats[col] = df[col].values
        feats[df.columns].to_csv(os.path.join(NOWWW_OUT, f"{key}_nowww.csv"), index=False)
        print(f"[i] {NAMES[key]}: {len(df)} rows, {int(has.sum())} prefixes removed", flush=True)
    rel = pd.read_csv(os.path.join(PROC, "vn_compphish.csv"), usecols=["url", "label"])
    has, y = rel["url"].astype(str).map(lambda u: bool(WWW.match(u))), rel["label"].map(is_phish)
    for cls, m in (("phishing", y), ("benign", ~y)):
        share.append({"corpus": "PhishVN (released)", "class": cls, "rows": int(m.sum()),
                      "www_share": float(has[m].mean())})
    pd.DataFrame(share).to_csv(os.path.join(NOWWW_OUT, "www_share.csv"), index=False)
    print(pd.DataFrame(share).round(3).to_string(index=False))


build()
nowww_corpora = [f"{NAMES[k]}={os.path.join(NOWWW_OUT, k + '_nowww.csv')}" for k in SOURCES]
with threadpool_limits(1, "blas"):
    for metric, model, suf in (("F1", "RandomForest", ""), ("ROC-AUC", "RandomForest", ""),
                               ("MCC", "RandomForest", ""), ("F1", "CatBoost", "_CatBoost")):
        run_cross_dataset(corpora=nowww_corpora, metric=metric, model=model,
                          out=os.path.join(NOWWW_OUT, f"cross_dataset_{metric}{suf}.csv"))
check_table("23 no www prefix", "nowww/www_share.csv", MET_TOL, path=os.path.join(NOWWW_OUT, "www_share.csv"))
check_table("23 no www prefix", "nowww/phishvn_nowww.csv", MET_TOL, path=os.path.join(NOWWW_OUT, "phishvn_nowww.csv"))
for f in ("cross_dataset_F1.csv", "cross_dataset_ROC-AUC.csv", "cross_dataset_MCC.csv"):
    check_matrix("23 no www prefix", f"nowww/{f}", 0.0, path=os.path.join(NOWWW_OUT, f))
report("23 no www prefix")

## Bước 24: Các con số tính từ kết quả

Phần lớn con số của bài là trung bình và độ lệch đọc thẳng từ các bảng đã so ở trên. Một số khác được tính thêm từ các bảng đó hoặc từ chính corpus: kiểm định có
cặp, bootstrap, tương quan, và vài phép đếm lại. Bước này tính lại chúng (từ các bảng notebook vừa tạo) và so với `results/p2_generated.json` và
`results/p2_shiftmatrix.json`, hai file bài ghi các con số đó ra.

Các kiểm định có cặp dùng `corrected_paired_t` và `bh_adjust` của Bước 1.

In [ ]:
from scipy import stats
from scipy.stats import spearmanr

ORDER = ["CatBoost", "MLP", "XGBoost", "LightGBM", "RandomForest", "HistGB", "LogReg"]
R = {}
rd = lambda name: pd.read_csv(f"{RES}/{name}")
strict = rd("p2_temporal_strict.csv")
bench = rd("p2_benchmark.csv")
ts = strict[strict.protocol == "temporal_strict"]


# CatBoost minus logistic regression under the phishing-temporal protocol, paired by seed, on all four metrics
def pair(metric):
    cb = ts[ts.family == "CatBoost"].sort_values("seed")[metric].to_numpy()
    lr = ts[ts.family == "LogReg"].sort_values("seed")[metric].to_numpy()
    return corrected_paired_t(cb - lr)


f1 = pair("F1")
R["strict_cb_minus_lr"] = {"F1": f1, "F1_ci95_half": stats.t.ppf(0.975, f1["k"] - 1) * f1["se"] if f1["se"] else 0.0,
                           **{m: pair(m) for m in ("PR-AUC", "ROC-AUC", "FPR@R0.90")}}
for m in ("F1", "PR-AUC", "ROC-AUC", "FPR@R0.90"):
    t = R["strict_cb_minus_lr"][m]
    print(f"CatBoost minus logistic regression, {m}: {t['mean']:+.4f} (corrected p={t['p']:.3g}, {t['wins']}/{t['k']} seeds positive)")


# the three same-corpus designs, family by family (F1 at 0.5), and the rows they rest on
def mean_f1(df, proto, fam):
    return float(df[(df.family == fam) & (df.protocol == proto)]["F1"].mean())


stage = {"Full": [mean_f1(bench, "random", f) for f in ORDER],
         "Dated": [mean_f1(strict, "random_same_rows", f) for f in ORDER],
         "Temp": [mean_f1(strict, "temporal_strict", f) for f in ORDER]}
is_ph = df_all["y"].eq(1)
R["decomposition"] = {"stage_f1": stage, "rows_full": len(df_all),
                      "rows_dated": int((~is_ph).sum() + (is_ph & df_all["date"].notna()).sum())}
print("best F1 by design (full random, dated random, phishing-temporal):", [round(max(v), 3) for v in stage.values()])


# the oracle maximum F1 read off each family's seed-mean PR curve
def max_f1_curve(curves):
    out = {}
    for fam, g in curves.groupby("family"):
        r, p_ = g["recall"].to_numpy(float), g["precision"].to_numpy(float)
        with np.errstate(divide="ignore", invalid="ignore"):
            f = np.where(r + p_ > 0, 2 * r * p_ / (r + p_), 0.0)
        out[fam] = float(np.nanmax(f))
    return out


cf, cs = rd("p2_pr_curves.csv"), rd("p2_pr_curves_strict.csv")
R["max_f1"] = {"Random (full)": max_f1_curve(cf[cf.protocol == "random"]),
               "Random (dated rows)": max_f1_curve(cs[cs.protocol == "random_same_rows"]),
               "Phishing-temporal": max_f1_curve(cs[cs.protocol == "temporal_strict"])}

In [ ]:
# the all-positive F1 floor 2p/(1+p) of each test set, against the best family
files = {"PhishVN": "vn_compphish.csv", "PhiUSIIL": "external/phiusiil_compphish.csv",
         "ISCXURL2016": "external/iscx_compphish.csv", "PhishStorm": "external/phishstorm_compphish.csv"}
mat = pd.read_csv(f"{RES}/cross_dataset_F1.csv", index_col=0)
rates, floor = {}, {}
for n in mat.columns:
    y_ = add_label(pd.read_csv(os.path.join("data/processed", files[n])))["y"]
    rates[n] = float(y_.mean())
    floor[n] = 2 * rates[n] / (1 + rates[n])
off = [(i, j) for i in mat.columns for j in mat.columns if i != j]


def dated_test_counts():
    ph = df_all[(df_all.y == 1) & df_all.date.notna()].sort_values("date")
    cut = int(len(ph) * 0.70)
    ph_tr, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    n_ph = int((~ph_te.rdom.isin(set(ph_tr.rdom))).sum())
    n_be_pool = int((df_all.y == 0).sum())
    n_be = int(np.mean([(np.random.RandomState(s).rand(n_be_pool) >= 0.70).sum() for s in range(5)]))
    return n_ph, n_be


n_ph, n_be = dated_test_counts()
dated_p = n_ph / (n_ph + n_be)
best = lambda df, proto: float(df[df.protocol == proto].groupby("family")["F1"].mean().max())
R["trivial_floor"] = {"positive_rate": rates, "all_positive_f1": floor,
                      "offdiag_f1_mean": float(np.mean([float(mat.loc[i, j]) for i, j in off])),
                      "offdiag_floor_mean": float(np.mean([floor[j] for _, j in off])),
                      "dated_test_phish": n_ph, "dated_test_benign": n_be,
                      "designs": [["Full corpus random", rates["PhishVN"], best(bench, "random")],
                                  ["Dated subset random", dated_p, best(strict, "random_same_rows")],
                                  ["Dated subset temporal", dated_p, best(strict, "temporal_strict")]]}
print("all-positive F1 floor per corpus:", {k: round(v, 3) for k, v in floor.items()})


# confusion counts at 90% recall for CatBoost, from the guarded test size and the seed-mean FPR
def random_arm_phish_test_mean():
    ph = df_all[(df_all.y == 1) & df_all.date.notna()].sort_values("date")
    cut = int(len(ph) * 0.70)
    ph_tr, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr.rdom))]
    n_be_pool = int((df_all.y == 0).sum())
    n = len(ph_tr) + len(ph_te)
    sizes = []
    for s in range(5):
        rng = np.random.RandomState(s)
        rng.rand(n_be_pool)  # the benign mask is drawn first
        sizes.append(int((~(rng.rand(n) < len(ph_tr) / n)).sum()))
    return int(round(np.mean(sizes)))


tp = round(0.90 * n_ph)
cells = {}
for proto in ("random_same_rows", "temporal_strict"):
    fp = round(strict[(strict.family == "CatBoost") & (strict.protocol == proto)]["FPR@R0.90"].mean() * n_be)
    cells[proto] = (fp, n_be - fp)
R["confusion"] = {"dated_test_phish": n_ph, "dated_test_benign": n_be, "random_arm_phish_test": random_arm_phish_test_mean(),
                  "tp": tp, "fn": n_ph - tp, "fp_tn": cells}

# CORAL cells that sit on the all-positive floor, and how many of those rank below chance
f_ = pd.read_csv(f"{RES}/cross_dataset_F1_coral.csv", index_col=0)
r_ = pd.read_csv(f"{RES}/cross_dataset_ROC-AUC_coral.csv", index_col=0)
const = {k: 2 * v / (1 + v) for k, v in {n: (rates["PhishVN"] if n == "PhishVN" else 0.5) for n in f_.columns}.items()}
deg, sub = [], 0
for i in f_.index:
    for j in f_.columns:
        if i != j and abs(f_.loc[i, j] - const[j]) < 0.005:
            deg.append([i, j, f_.loc[i, j], r_.loc[i, j]])
            sub += r_.loc[i, j] < 0.5
R["coral_degeneracy"] = {"degenerate_cells": deg, "below_chance": int(sub), "offdiag_cells": len(f_.index) * (len(f_.columns) - 1)}
print(f"CORAL: {len(deg)} of {R['coral_degeneracy']['offdiag_cells']} off-diagonal cells sit on the all-positive F1, {int(sub)} of them below chance")

# the corpus's own train/val/test split: phishing rows and all rows, by evidence tier
u = pd.read_csv("data/processed/dataset_url.csv", usecols=["label", "tier", "split"])
u["is_phish"] = u["label"].astype(str).eq("phishing")
R["official_split_priors"] = {
    name: {sp: [int(d[d["split"].astype(str).eq(sp)]["is_phish"].sum()), int(d["split"].astype(str).eq(sp).sum())]
           for sp in ("train", "val", "test")}
    for name, d in (("All tiers", u), ("Gold+silver", u[u["tier"].astype(str).isin(["gold", "silver"])]),
                    ("Gold only", u[u["tier"].astype(str).eq("gold")]))}

In [ ]:
# the dated phishing rows: how many come from each feed, where the cut falls, and how the reconstructed dates
# compare with the national feed's (from the first-seen validation summary)
src = pd.read_csv("data/processed/dataset_url.csv", usecols=["url_norm", "source"], dtype=str)
dd = df_all.merge(src.rename(columns={"url_norm": "url"}).drop_duplicates("url"), on="url", how="left")
ph = dd[(dd.y == 1) & dd.date.notna()].sort_values("date").reset_index(drop=True)
tr, te, _ = split_phishing(ph, 0.70)
ncsc = lambda d: 100 * float((d.source == "tinnhiemmang").mean())
R["dates"] = {"dated_phish": len(ph), "chongluadao": int((ph.source == "chongluadao").sum()), "train": len(tr), "test": len(te),
              "test_from_2024": int((te.date >= pd.Timestamp("2024-01-01")).sum()), "test_first_day": str(te.date.min().date()),
              "ncsc_share_train_pct": ncsc(tr), "ncsc_share_test_pct": ncsc(te)}
print("dated phishing rows:", R["dates"])

# the one-sided guard: how many benign test rows share a registrable domain with benign training rows, how many phishing
# rows the guard removes from the random control, and the two test priors
ph = df_all[(df_all.y == 1) & df_all.date.notna()].sort_values("date").reset_index(drop=True)
be = df_all[df_all.y == 0].reset_index(drop=True)
tr, te, _ = split_phishing(ph, 0.70)
shared, drop_, p_t, p_g = [], [], [], []
for s in range(5):
    rng = np.random.RandomState(s)
    bm = benign_mask(be, rng, 0.70)
    shared.append(be[~bm].rdom.isin(set(be[bm].rdom)).mean())
    pool = pd.concat([tr, te])
    pm = rng.rand(len(pool)) < len(tr) / (len(tr) + len(te))
    g = pool[~pm].rdom.isin(set(pool[pm].rdom))
    nb = int((~bm).sum())
    drop_.append(g.mean())
    p_t.append(len(te) / (len(te) + nb))
    p_g.append(int((~g).sum()) / (int((~g).sum()) + nb))
R["guard_macros"] = {"benign_shared": shared, "guard_drop": drop_, "prior_temporal": p_t, "prior_guarded": p_g}
print(f"benign test rows sharing a domain with training: {100 * np.mean(shared):.0f}%; phishing rows the guard drops: {100 * np.mean(drop_):.1f}%")

# the protocol step before and after the guard is applied to the control as well
g_ = rd("p2_temporal_strict_guarded.csv")
mean_m = lambda df, proto, fam, metric: float(df[(df.family == fam) & (df.protocol == proto)][metric].mean())
unguarded = {f: mean_m(strict, "random_same_rows", f, "F1") for f in ORDER}
guarded = {f: mean_m(g_, "random_same_rows_guarded", f, "F1") for f in ORDER}
temporal = {f: mean_m(strict, "temporal_strict", f, "F1") for f in ORDER}
d_old = {f: temporal[f] - unguarded[f] for f in ORDER}
d_new = {f: temporal[f] - guarded[f] for f in ORDER}
lp = rd("p2_dup_leakage_protocols.csv")
pm_, pg_ = rd("p2_temporal_strict_guarded_pm.csv"), rd("p2_temporal_strict_guarded_pm_bgroup.csv")
step_ = lambda df, f, m: mean_m(df, "temporal_strict", f, m) - mean_m(df, "random_same_rows_guarded", f, m)
d_roc = {f: step_(pm_, f, "ROC-AUC") for f in ORDER}
d_roc_u = {f: mean_m(strict, "temporal_strict", f, "ROC-AUC") - mean_m(strict, "random_same_rows", f, "ROC-AUC") for f in ORDER}
boost = [f for f in ORDER if f != "LogReg"]
R["guard_control"] = {
    "delta_old": d_old, "delta_new": d_new,
    "regdom_shared_random": lp[lp.protocol == "random_same_rows"]["phish_test_regdom_shared_rate"].mean(),
    "regdom_shared_temporal": lp[lp.protocol == "temporal_strict"]["phish_test_regdom_shared_rate"].mean(),
    "guard_shift": float(np.mean([guarded[f] - unguarded[f] for f in ORDER])),
    "catboost_prauc_old": mean_m(strict, "temporal_strict", "CatBoost", "PR-AUC") - mean_m(strict, "random_same_rows", "CatBoost", "PR-AUC"),
    "catboost_prauc_new": mean_m(strict, "temporal_strict", "CatBoost", "PR-AUC") - mean_m(g_, "random_same_rows_guarded", "CatBoost", "PR-AUC"),
    "spearman_old_new": float(spearmanr([d_old[f] for f in ORDER], [d_new[f] for f in ORDER]).statistic),
    "pm_f1": {f: step_(pm_, f, "F1") for f in ORDER}, "pm_roc": d_roc, "pm_fpr": {f: step_(pm_, f, "FPR@R0.90") for f in ORDER},
    "unguarded_roc": d_roc_u, "pm_bgroup_roc": {f: step_(pg_, f, "ROC-AUC") for f in ORDER},
    "roc_share_removed": 1 - np.mean([d_roc[f] for f in boost]) / np.mean([d_roc_u[f] for f in boost])}
print(f"share of the ROC-AUC protocol step the guard and prior matching remove: {100 * R['guard_control']['roc_share_removed']:.0f}%")

In [ ]:
# the character CNN against each tabular family, paired on identical rows, BH across all 42 comparisons
cnn = rd("p2_charcnn.csv")


def paired_cnn(metric, proto, fam):
    c = cnn[cnn.protocol == proto].set_index("seed")[metric]
    t = strict[(strict.protocol == proto) & (strict.family == fam)].set_index("seed")[metric]
    seeds = sorted(set(c.index) & set(t.index))
    d = np.array([c[i] - t[i] for i in seeds], float)
    return d.mean(), corrected_paired_t(d)["p"]


fams = sorted(strict[strict.protocol == "temporal_strict"].family.unique())
res, ps = [], []
for fam in fams:
    for m in ("F1", "PR-AUC", "FPR@R0.90"):
        for proto in ("temporal_strict", "random_same_rows"):
            delta, p_ = paired_cnn(m, proto, fam)
            res.append((fam, m, proto, delta))
            ps.append(p_)
qs, rej = bh_adjust(ps)
best_fam = max(fams, key=lambda f: strict[(strict.protocol == "temporal_strict") & (strict.family == f)]["F1"].mean())


def dproto(fam):
    src_ = cnn if fam == "CharCNN" else strict[strict.family == fam]
    return src_[src_.protocol == "temporal_strict"]["F1"].mean() - src_[src_.protocol == "random_same_rows"]["F1"].mean()


def spread_gain(proto):
    tab = strict[strict.protocol == proto].groupby("family")["F1"].mean()
    return [cnn[cnn.protocol == proto]["F1"].mean() - tab.max(), tab.max() - tab.min()]


R["charcnn"] = {"paired": [[f, m, pr, d, p_, q_] for (f, m, pr, d), p_, q_ in zip(res, ps, qs)],
                "n_significant": int(sum(rej)), "best_tabular": best_fam,
                "vs_best_temporal": {m: paired_cnn(m, "temporal_strict", best_fam)[0] for m in ("F1", "PR-AUC", "FPR@R0.90")},
                "protocol_drop_cnn": dproto("CharCNN"), "protocol_drop_tabular": {f: dproto(f) for f in fams},
                "gain_and_spread": {p_: spread_gain(p_) for p_ in ("temporal_strict", "random_same_rows")}}
print(f"character CNN vs tabular families: {int(sum(rej))} of {len(ps)} paired comparisons significant after BH")

# the benign-source probe: the swap's effect against each family's canonical temporal score
pr_ = rd("p2_source_probe.csv")
swap = lambda fam, tb, metric="F1": float(pr_[(pr_.family == fam) & (pr_.train_benign == tb)][metric].mean())
canon = lambda fam, metric="F1": float((cnn[cnn.protocol == "temporal_strict"] if fam == "CharCNN" else
                                        strict[(strict.protocol == "temporal_strict") & (strict.family == fam)])[metric].mean())
R["source_probe"] = {"cnn_tranco": swap("CharCNN", "tranco") - canon("CharCNN"), "catboost_tranco": swap("CatBoost", "tranco") - canon("CatBoost"),
                     "logreg_tranco": swap("LogReg", "tranco") - canon("LogReg"), "catboost_tranco_prauc": swap("CatBoost", "tranco", "PR-AUC"),
                     "cnn_tinnhiem": swap("CharCNN", "tinnhiem") - canon("CharCNN"), "catboost_tinnhiem": swap("CatBoost", "tinnhiem") - canon("CatBoost")}

# stacking against CatBoost under the phishing-temporal protocol (BH across the six ensembles), and the 20-seed crossover
STACKS = [("Stacking", "Stack[RF+XGB]"), ("Stacking+HFS", "Stack[RF+XGB]+HFS"), ("Stack[XGB+LGBM]", "Stack[XGB+LGBM]"),
          ("Stack[CB+HGB]", "Stack[CB+HGB]"), ("Stack[CB+MLP]", "Stack[CB+MLP]"), ("Stack[CB+HGB+LR+MLP]", "Stack[CB+HGB+LR+MLP]"),
          ("Stack[CB+LR]", "Stack[CB+LR]")]
df5 = pd.concat([rd("p2_stacking_baseline.csv").replace({"family": {v: k for k, v in STACKS[:2]}}), rd("p2_stacking_combos.csv")])


def paired_stack(a_df, b_df, fam_a, fam_b, proto, k_col="PR-AUC"):
    a = a_df[(a_df.family == fam_a) & (a_df.protocol == proto)].set_index("seed")[k_col]
    b = b_df[(b_df.family == fam_b) & (b_df.protocol == proto)].set_index("seed")[k_col]
    return corrected_paired_t((b.sort_index() - a.sort_index()).dropna().values, test_frac=0.30)


sfams = [k for k, _ in STACKS if k != "Stacking+HFS"]
sres = {f: paired_stack(strict, df5, "CatBoost", f, "temporal_strict") for f in sfams}
sq = dict(zip(sfams, bh_adjust([sres[f]["p"] for f in sfams])[0]))
cb20, st20 = rd("p2_temporal_strict_cb_k20.csv"), rd("p2_stacking_cblr_k20.csv")
t20 = paired_stack(cb20, st20, "CatBoost", "Stack[CB+LR]", "temporal_strict")
r20 = paired_stack(cb20, st20, "CatBoost", "Stack[CB+LR]", "random_same_rows")
tq, rq = bh_adjust([t20["p"], r20["p"]])[0]
R["stacking"] = {"k5_vs_catboost": {f: {**sres[f], "q": sq[f]} for f in sfams},
                 "k20_temporal": {**t20, "q": tq}, "k20_random": {**r20, "q": rq}}
for f in sfams:
    print(f"{f:22s} minus CatBoost, PR-AUC {sres[f]['mean']:+.4f} (BH q={sq[f]:.3f})")

In [ ]:
# can drift be forecast? decay slopes against the protocol gap; the block shift matrix; lexical novelty; FCTS
dec = rd("p2_forecastability_decay.csv")
gg_ = strict.groupby(["family", "protocol"])["PR-AUC"].mean().unstack()
gap = gg_["random_same_rows"] - gg_["temporal_strict"]
rhos = []
for origin in sorted(dec.origin.unique()):
    for metric in ("PR-AUC", "ROC-AUC"):
        sub_ = dec[dec.origin == origin]
        slopes = {}
        for fam in sub_.family.unique():
            w = sub_[sub_.family == fam].groupby("win")[["horizon_d", metric]].mean()
            slopes[fam] = np.polyfit(w["horizon_d"], w[metric], 1)[0]
        sl = pd.Series(slopes)
        rho, p_ = spearmanr(-sl[gap.index], gap)
        rhos.append([origin, metric, rho, p_])
R["forecast_decay_spearman"] = rhos
win_means = dec[(dec.origin == 0.50) & (dec.family == "CatBoost")].groupby("win")["PR-AUC"].mean()
mx = rd("p2_forecastability_shiftmatrix.csv")
internal = mx[mx.a.str.startswith("W") & mx.b.str.startswith("W")]
adj = internal[internal.b.str[1].astype(int) - internal.a.str[1].astype(int) == 1]
row_ = lambda a, b: mx[(mx.a == a) & (mx.b == b)].iloc[0]
nov = rd("p2_forecastability_novelty.csv")
cbn = nov[nov.family == "CatBoost"]
R["forecast_other"] = {"catboost_win_swing": win_means.max() - win_means.min(), "adjacent_auc_mean": adj.auc.mean(),
                       "auc_W4_TEST": row_("W4", "TEST").auc, "auc_W1_W4": row_("W1", "W4").auc, "auc_W1_TEST": row_("W1", "TEST").auc,
                       "novelty_auc_all": cbn.auc_all.mean(), "novelty_auc_phish": cbn.auc_phish.mean(),
                       "novelty_seeds_below_half": int((cbn.auc_all < 0.5).sum())}
fc = rd("p2_fcts.csv")
fct = fc[fc.protocol == "temporal_strict"]
piv = fct.pivot(index="seed", columns="variant", values="PR-AUC")
R["fcts"] = {"forward_minus_shuffled": corrected_paired_t(piv.forward_chained - piv.shuffled_order),
             "forward_minus_random": corrected_paired_t(piv.forward_chained - piv.random_fold),
             "coef_ratio_cb_lr": (fct.groupby("variant").coef_CB.mean() / fct.groupby("variant").coef_LR.mean()).to_dict()}
print("decay slope vs protocol gap, Spearman by origin and metric:", [(o, m, round(r, 2)) for o, m, r, _ in rhos])

# the block shift matrix summarised as in the paper's figure: elapsed time against separability
rows_ = rd("p2_forecastability_shiftmatrix.csv").to_dict("records")
ADJACENT = (("W1", "W2"), ("W2", "W3"), ("W3", "W4"))
adj_r = [r for r in rows_ if (r["a"], r["b"]) in ADJACENT]
bnd_r = [r for r in rows_ if r["b"] == "TEST"]
rho, p_ = spearmanr([r["delta_days"] for r in rows_], [r["auc"] for r in rows_])
shiftmatrix = {"n": len(rows_), "rho": float(rho), "p": float(p_), "adj_mean": sum(r["auc"] for r in adj_r) / len(adj_r),
               "adj_lo": min(r["auc"] for r in adj_r), "adj_hi": max(r["auc"] for r in adj_r),
               "bnd_mean": sum(r["auc"] for r in bnd_r) / len(bnd_r)}

In [ ]:
# what the deduplicated corpus moves (step 22)
DEDUP = "data/processed/p2/dedup_audit/"
fam_means = lambda path, proto, met="F1": (lambda d: d[d.protocol == proto].groupby("family")[met].mean())(pd.read_csv(path))


def dated_of(path):
    d = pd.read_csv(path, usecols=["label", "collected_at"], low_memory=False)
    t = d["collected_at"].astype(str).str.strip().str.slice(0, 10)
    ok = (pd.to_datetime(t, format="%d/%m/%Y", errors="coerce").fillna(pd.to_datetime(t, format="%Y-%m-%d", errors="coerce"))).notna()
    return d, ok


uu, du = dated_of("data/processed/dataset_url.csv")
vv, dv = dated_of(DEDUP + "dataset_url_dedup.csv")
php = vv.label.astype(str).str.lower() == "phishing"
full1, dr1, t1 = (fam_means(DEDUP + "p2_benchmark.csv", "random"), fam_means(DEDUP + "p2_temporal_strict.csv", "random_same_rows"),
                  fam_means(DEDUP + "p2_temporal_strict.csv", "temporal_strict"))
stk = pd.concat([full1, dr1, t1], axis=1)
r0 = pd.read_csv(f"{RES}/cross_dataset_ROC-AUC.csv", index_col=0)
r1 = pd.read_csv(DEDUP + "cross_dataset_ROC-AUC.csv", index_col=0)
e = np.eye(len(r0), dtype=bool)
flips = [[i, j] for i in r0.index for j in r0.columns if i != j and (r0.loc[i, j] < 0.5) != (r1.loc[i, j] < 0.5)]
gap_of = lambda p_: (lambda m_: m_[e].mean() - m_[~e].mean())(pd.read_csv(p_, index_col=0).values.astype(float))
R["dedup_sensitivity"] = {
    "rows_released": len(uu), "rows_dedup": len(vv), "dated_dropped": int(du.sum() - dv.sum()), "undated_phish_left": int((php & ~dv).sum()),
    "family_move_max": float((stk.max(axis=1) - stk.min(axis=1)).max()),
    "offdiag_roc_released": r0.values[~e].mean(), "offdiag_roc_dedup": r1.values[~e].mean(),
    "below_chance_released": int((r0.values[~e] < 0.5).sum()), "below_chance_dedup": int((r1.values[~e] < 0.5).sum()),
    "flipped_cells": flips,
    "rf_f1_gap": [gap_of(f"{RES}/cross_dataset_F1.csv"), gap_of(DEDUP + "cross_dataset_F1.csv")],
    "catboost_f1_gap": [gap_of(f"{RES}/cross_dataset_F1_CatBoost.csv"), gap_of(DEDUP + "cross_dataset_F1_CatBoost.csv")]}
print(f"deduplicated corpus: {len(vv):,} of {len(uu):,} rows; {len(flips)} transfer cells change side of chance")

check_json("24 derived numbers", R, "results/p2_generated.json", tol=MET_TOL)
check_json("24 derived numbers", shiftmatrix, "results/p2_shiftmatrix.json", tol=MET_TOL)
report("24 derived numbers")

## Bước 25: Tổng hợp

Bảng dưới gom mọi phép kiểm theo bước.

In [ ]:
tbl = pd.DataFrame(CHECKS)
display(tbl.groupby("step", sort=False)["match"].agg(checks="size", matched="sum"))
print(f"total: {int(tbl['match'].sum())}/{len(tbl)} checks match")